# LSPV-Bench v2.0.4 — Which data-scarcity remedy should a new PV plant use?

**Research question.** Under an *equal training-update budget*, which remedy gives a newly commissioned PV plant the most forecast skill over smart persistence with 14, 60 or 365 days of local data: learned generative augmentation (DDPM, as in the base paper), simple replication or jitter, physics-simulated pre-training (pvlib) or transfer from a data-rich array? Does the answer change between an LSTM and a tuned LightGBM, and at what data length does each remedy stop helping?

**Base paper.** Demir A., Najaf A. *Diffusion-Augmented LSTM for Improved Generalization in Cross-Site Solar Power Forecasting.* Int. Trans. Electr. Energy Syst. 2026:6970304. doi:10.1155/etep/6970304. Used as a baseline only (no code or text copied).

**Gap (from the Stage 1 error register, 9 Nov 2026).** The base paper's DDPM gain is confounded with 50% more gradient updates (E4), synthetic data also entered the validation set (E3), there is no persistence reference (E2), the cross-site target is a cumulative daily yield (E1), metrics include night zeros and MAPE (E9, E10), and the statistics rest on single seeds or "one valid pair" (E5, E24). A Scopus-indexed search (LeapSpace S1–S6, 138 unique records, 9 Nov 2026) found no study that compares generative augmentation, simple controls, physics pre-training and transfer under one protocol; this is "no overlap found in a dated search", not proof of a gap.

## Hypotheses (two-sided; pre-declared before any confirmatory data are read)
| ID | Hypothesis | Test | Gate |
|---|---|---|---|
| H1 | DDPM augmentation (A3) differs from replication (A1) at each data length, under equal updates | Diebold–Mariano on daily losses, Holm; TOST at ±2% RMSE | SUPPORTED if Holm p < 0.05; EQUIVALENT if the 90% CI of the RMSE ratio lies within ±2% |
| H2 | Transfer (A5) differs from target-only (A0) at 14 and 60 days | DM, Holm | Holm p < 0.05 |
| H3 | The ranking of remedies changes with data length | rank at 14 d vs 365 d | different top arm on ≥ 2 of 3 targets |
| H4 | The base paper's DDPM gain reproduces under its own protocol and shrinks under equal updates | replication on the Kaggle plants | gain(base) > 0 and gain(equal) < gain(base) |
| D2 | The effect of a remedy depends on the model (LSTM vs LightGBM) | interaction of skill gains, block bootstrap | 95% CI excludes 0 for any remedy |
| D7 | Each remedy has a data length beyond which it stops helping | crossover table | reported for every remedy × model |

## Novelty-to-evidence matrix
| Claim | Experiment | Metric | Table | Acceptance |
|---|---|---|---|---|
| DDPM vs simple controls at equal budget | A0–A3, equal updates | daytime RMSE, skill vs SP | `tables/statistical_tests.csv` | H1 gate |
| Outside information (transfer, physics) vs augmentation | A4, A5 vs A0–A3 | skill vs SP by length | `tables/main_results.csv` | H2, H3 gates |
| Model dependence | LSTM vs LightGBM | interaction | `tables/interaction_backbone.csv` | D2 gate |
| When remedies stop helping | 14/60/365 d | gain vs A0 with CI | `tables/crossover_length.csv` | D7 |
| Base-paper replication | Kaggle plants, base vs equal protocol | RMSE | `tables/replication_base_paper.csv` | H4 |

## Arms
A0 target-only · A1 replication to 1.5× · A2 jitter (+50% noisy copies, σ = 0.05 SD) · A3 DDPM (+50% samples, conditional on time features, trained on target history only) · A4 physics pre-training (pvlib simulation of the target array from weather data before the history window) · A5 transfer (pre-training on a source array, data ≤ 2021-12-31) · A6 oracle (all pre-origin target data; Phase 0 headroom only). References: smart persistence (clear-sky index) and the two backbones themselves.

**Equal update budget.** Every arm of a split gets U = max(50 × ceil(n_real/32), 4000) LSTM updates (the 4000 floor fixes under-training at 14–60 days) and the same number of LightGBM tuning trials. Generator and pre-training costs are logged separately in `tables/compute_cost.csv`.

**Inputs.** 8 lags of the clear-sky index (2 h), time of day and clear-sky power at the target time (per-horizon LightGBM), or at the first horizon (two-output LSTM and DDPM). No day-of-year features: they cannot generalise from 14 days of history to a 90-day test window. Training loss is weighted by (clear-sky power / rating)² so the fit matches the kW evaluation.

## Modes
| Mode | Purpose | Scope | Publication |
|---|---|---|---|
| SMOKE_TEST | code plumbing on a small real subset | 1 target, 1 pilot origin (2022-02-01, after the site-wide outage ending 7 Jan 2022), 14 days, 1 seed | **NOT FOR PUBLICATION** |
| STANDARD_RUN | declared pilot / Phase 0 | 3 targets, 1 pilot origin (2022-07-01), 14/60/365 d, 2 seeds | exploratory |
| FULL_PAPER_RUN | confirmatory | 3 targets, origins 2023-01-01 and 2023-07-01, 14/60/365 d, 5 seeds | manuscript-eligible only if the software gate passes |

Confirmatory test data (2023) are never read in SMOKE_TEST or STANDARD_RUN.

**How to run (Colab, ajaffi1@gmail.com).** 1) Runtime → Change runtime type → T4 GPU. 2) In cell 1 set `MODE` (`STANDARD_RUN` or `FULL_PAPER_RUN`). 3) Runtime → Run all. 4) Data: put **`general.zip`** or **`general.7z`** in `MyDrive/RESEARCH/LSPV_Bench/` (Drive is on by default in Colab; approve the Drive sign-in once). Without it, cell 4 shows **Choose Files** for that one file. Nothing else is ever asked for. If a required file is missing, the run stops within minutes with `DATA INTAKE STOPPED` and the exact file names. 5) Send Claude the lines under `EARLY-OUTPUT CHECK`, then let it run. STANDARD and FULL stop at once if the runtime has no GPU. FULL does not rerun the base-paper replication (H4 comes from STANDARD); progress lines and Drive checkpoints cover every long stage. Reproduction check after FULL: in a fresh session, upload the FULL results ZIP and set `REPRO_REF_ZIP` in cell 1 to its path; the notebook reruns one split from scratch and compares.

**Reproducibility warning.** Results depend on the DKASC files you upload, their timestamp convention and array metadata; every choice is logged in `data_audit/` and `logs/`. Run account for this Project: ajaffi1@gmail.com.


In [ ]:
# ===== CELL 1 — CONFIGURATION (edit MODE only, unless told otherwise) =====
import os, sys, json, math, time, hashlib, shutil, zipfile, glob, re, random, platform, warnings, traceback
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_VERSION = "2.0.5"
FORMAT_VERSION = 1                      # bump when the task-file format changes
MODE = "SMOKE_TEST"                     # SMOKE_TEST | STANDARD_RUN | FULL_PAPER_RUN
MODE = os.environ.get("LSPV_MODE", MODE)
assert MODE in ("SMOKE_TEST", "STANDARD_RUN", "FULL_PAPER_RUN"), f"Unknown MODE {MODE}"
REPLICATION_ONLY = False               # True: run only the base-paper replication (H4); main benchmark is skipped (v2.0.5)
REPLICATION_ONLY = os.environ.get("LSPV_REPLICATION_ONLY", str(REPLICATION_ONLY)) == "True"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
RUN_ACCOUNT = "ajaffi1@gmail.com"       # JAFFARPROJECTS run account
USE_DRIVE = IN_COLAB                    # JAFFARPROJECTS default: in Colab, mount Drive, look for general.zip/.7z there, copy checkpoints and results to Drive
DRIVE_DIR = "/content/drive/MyDrive/RESEARCH/LSPV_Bench"
DATA_DIR = Path(os.environ.get("LSPV_DATA_DIR", "/content/lspv_data" if IN_COLAB else "./lspv_data"))   # general.zip is unpacked here

# ---- DATA: one file, general.zip (GENERAL.ZIP RULE). The notebook asks for nothing else. ----
GENERAL_ZIP = "general.zip"            # general.7z is accepted too (unpacked with 7z / py7zr)
# Exact file names, matched anywhere inside general.zip. role tells the loader what each file is.
REQUIRED_FILES = {
    "106-Site_DKA-M5_C-Phase.csv":     dict(role="array",    use="target array (11 BP Solar, 4.95 kW)"),
    "67-Site_DKA-M8_A-Phase.csv":      dict(role="array",    use="transfer source (21 Evergreen, 4.92 kW)"),
    "68-Site_DKA-M8_C-Phase.csv":      dict(role="array",    use="target array (20 Sungrid, 5.04 kW)"),
    "90-Site_DKA-M3_A-Phase.csv":      dict(role="array",    use="target array (14 Kyocera, 5.4 kW)"),
    "101-Site_DKA-WeatherStation.csv": dict(role="weather",  use="physics arm A4 (irradiance, temperature)"),
    "array_metadata.csv":              dict(role="metadata", use="rating, tilt, azimuth of each array"),
}
OPTIONAL_FILES = {
    "5-Site_DG-PV1-DB-DG-M1A.csv":     dict(role="yulara_array", use="H4 Yulara replication"),
    "Plant_1_Generation_Data.csv":     dict(role="kaggle_gen",   use="H4 Kaggle replication"),
    "Plant_1_Weather_Sensor_Data.csv": dict(role="kaggle_wx",    use="H4 Kaggle replication"),
    "Plant_2_Generation_Data.csv":     dict(role="kaggle_gen",   use="H4 Kaggle replication"),
    "Plant_2_Weather_Sensor_Data.csv": dict(role="kaggle_wx",    use="H4 Kaggle replication"),
}
# Workspace tests only (never set in Colab): replace the lists with the synthetic test file names.
if os.environ.get("LSPV_FILES_JSON"):
    _fj = json.loads(Path(os.environ["LSPV_FILES_JSON"]).read_text()); REQUIRED_FILES, OPTIONAL_FILES = _fj["required"], _fj["optional"]
DATASET_SOURCES = {   # provenance: where each family comes from and its terms (licence text is not copied; "not stated" where none was found)
    "DKASC": dict(page="https://dkasolarcentre.com.au/locations/alice-springs", terms="see the DKASC website terms; licence not stated in the notebook"),
    "Kaggle": dict(page="https://www.kaggle.com/datasets/anikannal/solar-power-generation-data", terms="see the Kaggle dataset page (licence shown there); not stated in the notebook"),
    "helper": dict(page="array_metadata.csv written by the authors from the DKASC array pages", terms="authors' file"),
}

# Sensitivity (pre-declared, exploratory): augmentation amount for the three augmentation arms, at the shortest length,
# first origin, first seed only. 0.50 is the main setting and is not repeated.
SENS_AUG_FRACS = [0.25, 1.00]
# Base-paper replication (H4). It copies the base paper's slow protocol (batch 16, up to 50 epochs), so its cost is capped:
# off in FULL_PAPER_RUN (H4 is reported from STANDARD_RUN), and Yulara uses at most 2 seeds (STANDARD v2.0.3: >3.5 h on a T4).
RUN_REPLICATION = {"SMOKE_TEST": True, "STANDARD_RUN": True, "FULL_PAPER_RUN": False}[MODE]
RUN_REPLICATION = os.environ.get("LSPV_REPLICATION", str(RUN_REPLICATION)) == "True"
YULARA_REP_SEEDS = 2
if REPLICATION_ONLY:                    # dedicated replication run: H4 is confirmatory in FULL_PAPER_RUN, with 3 Yulara seeds
    RUN_REPLICATION, YULARA_REP_SEEDS = True, 3
# Reproduction gate: path to an earlier results ZIP of the SAME mode and configuration. When set, the notebook reruns only
# the first target × first origin × shortest length × first seed from scratch in a clean session and compares forecasts.
REPRO_REF_ZIP = ""                    # e.g. "/content/LSPV_results_FULL_PAPER_RUN_v2.0.3.zip" (upload it first)
REPRO_REF_ZIP = os.environ.get("LSPV_REPRO_REF", REPRO_REF_ZIP)
REPRO_TOL = 0.02                      # max relative RMSE difference per task (GPU non-determinism allowance)
OUT_ROOT = Path(os.environ.get("LSPV_OUT", "/content/lspv" if IN_COLAB else "./lspv_out"))
FAULT = os.environ.get("LSPV_FAULT", "")   # fault-injection hook used only by the pre-delivery tests

# Site: DKASC, Alice Springs (Desert Knowledge Precinct). Local standard time, no daylight saving.
SITE = dict(name="DKASC Alice Springs", lat=-23.762, lon=133.875, alt=546, tz="Australia/Darwin")
YULARA = dict(name="DKASC Yulara", lat=-25.24, lon=130.99, alt=508, tz="Australia/Darwin")
# Optional: put array details in DATA_DIR/array_metadata.csv with columns
# file_key, rating_kw, tilt_deg, azimuth_deg, technology  (file_key = part of the CSV file name)
DEFAULT_TILT, DEFAULT_AZIMUTH = 20.0, 0.0   # pvlib convention: azimuth 0 = north (correct for the southern hemisphere)

ARMS = ["A0_target", "A1_replicate", "A2_jitter", "A3_ddpm", "A4_physics", "A5_transfer"]
BACKBONES = ["lstm", "lgbm"]
HORIZONS = [1, 4]                      # 15-min steps ahead: 15 min and 60 min
LAGS = 8                               # 2 h of clear-sky index history
EQUIV_MARGIN = 0.02                    # ±2% RMSE equivalence margin (pre-declared)
SOURCE_END = "2021-12-31 23:59"        # transfer source data end
PHYS_START = "2018-01-01"              # physics simulation window start
VAL_FRAC = 0.20                        # last 20% of history = validation (real data only)
BATCH = 32
# Equal update budget per split: U = max(e_base × ceil(n_real/BATCH), u_min), identical for every arm of that split.
# u_min exists because the base paper's 50 epochs leave the LSTM under-trained on 14-60 days of data
# (workspace plumbing check, synthetic data: 750 updates → 0.37 kW RMSE, 3000 → 0.19 kW).
AUG_FRAC = 0.50                        # +50% synthetic / copied rows, as in the base paper
JITTER_SIGMA = 0.05

MODE_CFG = {
    "SMOKE_TEST":     dict(n_targets=1, origins=["2022-02-01"], lengths=[14], seeds=[0], test_days=14,
                           e_base=10, u_min=300, lgbm_trials=3, lgbm_rounds=300, ddpm_iters=300, pre_epochs=2,
                           oracle=True, rep_epochs=3, rep_seeds=[0], boot=200, max_date="2022-12-31 23:59"),
    "STANDARD_RUN":   dict(n_targets=3, origins=["2022-07-01"], lengths=[14, 60, 365], seeds=[0, 1], test_days=90,
                           e_base=50, u_min=4000, lgbm_trials=10, lgbm_rounds=2000, ddpm_iters=3000, pre_epochs=10,
                           oracle=True, rep_epochs=50, rep_seeds=[0, 1], boot=1000, max_date="2022-12-31 23:59"),
    "FULL_PAPER_RUN": dict(n_targets=3, origins=["2023-01-01", "2023-07-01"], lengths=[14, 60, 365], seeds=[0, 1, 2, 3, 4],
                           test_days=90, e_base=50, u_min=4000, lgbm_trials=20, lgbm_rounds=2000, ddpm_iters=5000, pre_epochs=20,
                           oracle=False, rep_epochs=50, rep_seeds=list(range(10)), boot=2000, max_date="2023-12-31 23:59"),
}
CFG = MODE_CFG[MODE]

# Test-only shrink hooks (never used in Colab runs): LSPV_SHORT=1 shrinks budgets for the workspace path tests.
if os.environ.get("LSPV_SHORT") == "1":
    CFG = dict(CFG, seeds=CFG["seeds"][:1], e_base=3, u_min=50, lgbm_trials=2, lgbm_rounds=60, ddpm_iters=60, pre_epochs=1,
               rep_epochs=2, rep_seeds=[0], boot=50, test_days=min(CFG["test_days"], 14), n_targets=min(CFG["n_targets"], 2))

# Design values: target/source selection and the physics window always use these, so a reproduction run picks the same
# arrays and the same pre-training data as the run it checks.
N_TARGETS_DESIGN, L_MAX_DESIGN = CFG["n_targets"], max(CFG["lengths"])
REPRO = bool(REPRO_REF_ZIP)
if REPRO:
    CFG = dict(CFG, n_targets=1, origins=CFG["origins"][:1], lengths=[min(CFG["lengths"])], seeds=CFG["seeds"][:1], oracle=False)

RUN_TAG = MODE + ("_REPRO" if REPRO else "") + ("_REPLICATION" if REPLICATION_ONLY else "")   # names outputs, checkpoints and the results ZIP
OUT = OUT_ROOT / f"project_outputs_{RUN_TAG}"
DIRS = {k: OUT / k for k in ["config", "data_audit", "checkpoints", "tasks", "predictions", "tables", "figures", "models", "logs"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
CFG_HASH = hashlib.sha256(json.dumps(dict(CFG, version=NOTEBOOK_VERSION, fmt=FORMAT_VERSION, arms=ARMS,
                                          backbones=BACKBONES, horizons=HORIZONS, lags=LAGS), sort_keys=True).encode()).hexdigest()[:12]
(DIRS["config"] / "config.json").write_text(json.dumps(dict(MODE=MODE, CFG=CFG, CFG_HASH=CFG_HASH, NOTEBOOK_VERSION=NOTEBOOK_VERSION,
                                                           SITE=SITE, ARMS=ARMS, BACKBONES=BACKBONES, HORIZONS=HORIZONS,
                                                           RUN_ACCOUNT=RUN_ACCOUNT), indent=2))
print(f"MODE={MODE} | version {NOTEBOOK_VERSION} | config hash {CFG_HASH} | outputs → {OUT}" + (" | REPRODUCTION CHECK" if REPRO else "") + (" | REPLICATION-ONLY RUN (H4)" if REPLICATION_ONLY else ""))
print(f"DATA: one file '{GENERAL_ZIP}' → unpacked to {DATA_DIR} | Drive {'ON: also looks in ' + DRIVE_DIR if USE_DRIVE else 'OFF'}")
print(f"  required ({len(REQUIRED_FILES)}): " + ", ".join(REQUIRED_FILES))
print(f"  optional ({len(OPTIONAL_FILES)}): " + ", ".join(OPTIONAL_FILES))
print("PUBLICATION STATUS:", "NOT FOR PUBLICATION (smoke test)" if MODE == "SMOKE_TEST"
      else "exploratory pilot" if MODE == "STANDARD_RUN" else "confirmatory (eligible only if the software gate passes)")


In [ ]:
# ===== CELL 2 — ENVIRONMENT, IMPORTS, PREFLIGHT =====
import importlib, subprocess
def _ensure(pkg, mod=None):
    mod = mod or pkg
    try:
        return importlib.import_module(mod)
    except ImportError:
        print(f"Installing {pkg} …")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        return importlib.import_module(mod)
for _p, _m in [("pvlib", None), ("lightgbm", None)]:
    _ensure(_p, _m)

import numpy as np
import pandas as pd
import scipy
import scipy.stats as sst
import lightgbm as lgb
import pvlib
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)

VERSIONS = {m.__name__: getattr(m, "__version__", "?") for m in [np, pd, scipy, lgb, pvlib, torch, matplotlib]}
VERSIONS["python"] = platform.python_version()
print("Versions:", VERSIONS)
(OUT / "requirements.txt").write_text("\n".join(f"{k}=={v}" for k, v in VERSIONS.items() if k != "python") + "\n")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9 if hasattr(os, "sysconf") else float("nan")
_disk_gb = shutil.disk_usage(str(OUT_ROOT)).free / 1e9
PREFLIGHT = dict(device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
                 ram_gb=round(_ram_gb, 1), disk_free_gb=round(_disk_gb, 1), cpus=os.cpu_count(), in_colab=IN_COLAB)
print("Preflight:", PREFLIGHT)
if MODE in ("STANDARD_RUN", "FULL_PAPER_RUN") and (IN_COLAB or os.environ.get("LSPV_FAKE_COLAB_CPU") == "1") and DEVICE.type != "cuda":
    raise SystemExit(f"🔴 {MODE} needs a GPU runtime but this runtime is CPU only (a runtime can come up as CPU even with T4 selected). "
                     "Runtime → Disconnect and delete runtime, then Runtime → Change runtime type → T4 GPU → Save, and Run all again. Nothing was run.")
if _disk_gb < 2:
    raise SystemExit(f"Only {_disk_gb:.1f} GB free disk; at least 2 GB needed.")
assert hasattr(pvlib.irradiance, "get_total_irradiance") and hasattr(pvlib.irradiance, "erbs"), "pvlib API missing functions"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
# ===== CELL 3 — UTILITIES: atomic writes, task ledger, progress/ETA, runtime log, checkpoints =====
T0 = time.time()

def atomic_write_text(path, text):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text); os.replace(tmp, path)

def save_csv(df, path, quiet=False):
    path = Path(path); tmp = path.with_suffix(".tmp.csv")
    df.to_csv(tmp, index=False); os.replace(tmp, path)
    if not quiet:
        print(f"  saved {path.relative_to(OUT)}  shape={df.shape}  {path.stat().st_size/1024:.1f} KB")

def save_json(obj, path):
    atomic_write_text(path, json.dumps(obj, indent=2, default=str))

class Ledger:
    """Every expected task is registered first; each ends COMPLETED, FAILED, SKIPPED or NOT_APPLICABLE with a reason."""
    def __init__(self):
        self.rows = {}
    def expect(self, tid, kind, required=True):
        self.rows.setdefault(tid, dict(task_id=tid, kind=kind, required=required, status="PENDING", reason="", seconds=np.nan, reused=False))
    def done(self, tid, seconds=np.nan, reused=False):
        self.rows[tid].update(status="COMPLETED", seconds=seconds, reused=reused)
    def fail(self, tid, reason):
        self.rows[tid].update(status="FAILED", reason=str(reason)[:300])
    def na(self, tid, reason):
        self.rows[tid].update(status="NOT_APPLICABLE", reason=reason)
    def df(self):
        return pd.DataFrame(list(self.rows.values()), columns=["task_id", "kind", "required", "status", "reason", "seconds", "reused"])
    def save(self):
        save_csv(self.df(), DIRS["logs"] / "task_ledger.csv", quiet=True)
LEDGER = Ledger()

RUNTIME = []
class Progress:
    def __init__(self, name, total, every=None):
        self.name, self.total, self.n, self.t0 = name, max(total, 1), 0, time.time()
        self.every = every or max(1, self.total // 10)
    def step(self, label=""):
        self.n += 1
        if self.n % self.every == 0 or self.n == self.total:
            el = time.time() - self.t0; eta = el / self.n * (self.total - self.n)
            print(f"  [{self.name}] {self.n}/{self.total}  elapsed {el/60:.1f} min  ETA {eta/60:.1f} min  {label}", flush=True)

def log_runtime(stage, seconds, units=1):
    RUNTIME.append(dict(mode=MODE, stage=stage, units=units, seconds=round(seconds, 2)))
    save_csv(pd.DataFrame(RUNTIME), DIRS["logs"] / f"runtime_{MODE}.csv", quiet=True)

def checkpoint_zip():
    """Zip task files + logs to the session disk (and Drive if enabled). Never triggers a browser download."""
    z = DIRS["checkpoints"] / "checkpoint.zip"; tmp = z.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in list(DIRS["tasks"].glob("*.npz")) + list(DIRS["tasks"].glob("REP_*.json")) + list(DIRS["logs"].glob("*.csv")):
            zf.write(p, p.relative_to(OUT))
    os.replace(tmp, z)
    if USE_DRIVE and IN_COLAB:
        shutil.copy(z, Path(DRIVE_DIR) / f"checkpoint_{RUN_TAG}.zip")

def restore_checkpoint():
    """Fill only missing task files from a checkpoint; reject a damaged archive; never overwrite newer work."""
    srcs = [DIRS["checkpoints"] / "checkpoint.zip"]
    if USE_DRIVE and IN_COLAB:
        srcs.append(Path(DRIVE_DIR) / f"checkpoint_{RUN_TAG}.zip")
    n = 0
    for z in srcs:
        if not z.exists():
            continue
        try:
            with zipfile.ZipFile(z) as zf:
                if zf.testzip() is not None:
                    raise zipfile.BadZipFile("CRC error")
                for name in zf.namelist():
                    if name.startswith("tasks/") and not (OUT / name).exists():
                        zf.extract(name, OUT); n += 1
        except (zipfile.BadZipFile, OSError) as e:
            print(f"  ⚠ checkpoint {z.name} rejected ({e}); recomputing instead")
    if n:
        print(f"  restored {n} missing task files from checkpoint")
    return n

def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def export_results_zip():
    """Results ZIP: tables, figures, forecasts, logs, audits (no task files). Used by the final cell and by an early stop."""
    zp = OUT_ROOT / f"LSPV_results_{RUN_TAG}_v{NOTEBOOK_VERSION}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        for sub in ["config", "data_audit", "tables", "figures", "predictions", "logs"]:
            for p in DIRS[sub].rglob("*"):
                if p.is_file() and ".tmp" not in p.name:
                    zf.write(p, Path(sub) / p.relative_to(DIRS[sub]))
        for p in ["RESULTS_AUDIT.md", "FEASIBILITY.md", "requirements.txt"]:
            if (OUT / p).exists():
                zf.write(OUT / p, p)
    assert zp.exists() and zp.stat().st_size > 0, "results ZIP missing"
    print(f"Results ZIP: {zp} ({zp.stat().st_size/1024:.0f} KB)")
    if USE_DRIVE and IN_COLAB:
        shutil.copy(zp, Path(DRIVE_DIR) / zp.name)
    if IN_COLAB and os.environ.get("LSPV_NO_DOWNLOAD") != "1":
        try:
            from google.colab import files as _cf
            _cf.download(str(zp))
        except Exception as e:
            print("download skipped:", e)
    return zp

class DataIntakeStopped(Exception):
    """Raised on purpose: the data are not usable, so the run stops before any long step."""

def stop_run(reasons, rows=None):
    """FAIL-FAST DATA INTAKE: save what is missing, write a FAIL audit and ZIP, then stop Run-all at once."""
    if rows is not None:
        save_csv(pd.DataFrame(rows), DIRS["data_audit"] / "missing_files.csv", quiet=True)
    save_csv(pd.DataFrame([dict(check="data intake", result="FAIL", detail="; ".join(reasons))]), DIRS["tables"] / "software_check.csv", quiet=True)
    atomic_write_text(OUT / "RESULTS_AUDIT.md", f"# RESULTS_AUDIT — {MODE}\n\nSOFTWARE: **FAIL** (DATA INTAKE STOPPED before any training) · version {NOTEBOOK_VERSION}\n\n"
                      + "\n".join(f"- {r}" for r in reasons) + "\n")
    log_runtime("data", time.time() - T0)
    print("\n" + "=" * 78 + "\n🔴 DATA INTAKE STOPPED — nothing was trained. Fix these and run again:")
    for r in reasons:
        print("   •", r)
    print("=" * 78)
    export_results_zip()
    raise DataIntakeStopped("; ".join(reasons))


In [ ]:
# ===== CELL 4 — DATA: acquisition, provenance, cleaning, validation, timestamp alignment, array selection =====
# Sources (download by hand; see the data request):
#   DKASC Alice Springs arrays + weather station: https://dkasolarcentre.com.au/locations/alice-springs?source=19
#   DKASC download portal (base paper's citation): https://dkasolarcentre.com.au/historical-data/download
#   Kaggle "Solar Power Generation Data": https://www.kaggle.com/datasets/anikannal/solar-power-generation-data
#   (optional) DKASC Yulara System 1: https://dkasolarcentre.com.au/source/yulara/yulara-1-fixed
# Put Yulara files in a sub-folder whose name contains "yulara". Never substitute synthetic data.
t_data = time.time()
# ---- GENERAL.ZIP intake: find it, or ask for it once by name; unpack once ----
def _is_general(name):
    n = Path(name).name.lower()
    return bool(re.fullmatch(r"general( \(\d+\))?\.(zip|7z)", n))     # general.zip or general.7z; Colab renames a second upload "general (1).zip"
def _unpack_7z(src, dst):
    """Unpack a .7z with the 7z tool if present (installed on demand in Colab), else with py7zr."""
    if shutil.which("7z") is None and IN_COLAB:
        subprocess.run("apt-get -qq install -y p7zip-full > /dev/null", shell=True)
    if shutil.which("7z"):
        r = subprocess.run(["7z", "x", "-y", f"-o{dst}", str(src)], capture_output=True, text=True)
        if r.returncode != 0:
            raise OSError(f"7z failed: {r.stderr[-300:]}")
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py7zr"], check=True)
        import py7zr
        with py7zr.SevenZipFile(src) as z:
            z.extractall(dst)
def _print_file_lists():
    print(f"  REQUIRED inside {GENERAL_ZIP}:"); [print(f"     - {n}  ({v['use']})") for n, v in REQUIRED_FILES.items()]
    print(f"  OPTIONAL inside {GENERAL_ZIP}:"); [print(f"     - {n}  ({v['use']})") for n, v in OPTIONAL_FILES.items()]
_marker = DATA_DIR / ".general_unpacked.json"
_gz = None
if not _marker.exists():
    _dirs = [Path("/content"), DATA_DIR] + ([Path(DRIVE_DIR)] if USE_DRIVE and IN_COLAB else [])
    _cands = [d / f"general.{ext}" for d in _dirs for ext in ("zip", "7z")]
    for _d in _dirs:
        _cands += sorted(_d.glob("general (*).zip")) + sorted(_d.glob("general (*).7z")) if _d.exists() else []
    _gz = next((c for c in _cands if c.exists()), None)
    if _gz is None and IN_COLAB:
        print(f"\n📦 Upload ONE file named '{GENERAL_ZIP}' or 'general.7z' (click 'Choose Files' below). It must contain:")
        _print_file_lists()
        print(f"  Large file (the DKASC set is ~600 MB)? Faster and safer: click 'Cancel upload', then either drag {GENERAL_ZIP} into the Files pane\n"
              f"  (left sidebar, top level = /content) or put it in {DRIVE_DIR} and set USE_DRIVE = True; then run this cell again.")
        from google.colab import files as _cf
        _up = _cf.upload()
        _ok = [n for n in _up if _is_general(n)]
        _bad = [n for n in _up if not _is_general(n)]
        if _bad:
            print(f"  ⚠ ignored (not named {GENERAL_ZIP}): {', '.join(_bad)}")
        if _ok:
            DATA_DIR.mkdir(parents=True, exist_ok=True)
            _gz = DATA_DIR / Path(_ok[0]).name.replace(" ", "_"); _gz.write_bytes(_up[_ok[0]])
    if _gz is None:
        stop_run([f"'general.zip' / 'general.7z' was not found or uploaded (looked in /content, {DATA_DIR}" + (f", {DRIVE_DIR}" if USE_DRIVE else "") + ")",
                  f"pack the required files into one archive named exactly 'general.zip' or 'general.7z' and run again"],
                 [dict(file=n, required=True, status="MISSING (no general.zip)") for n in REQUIRED_FILES])
    try:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        if _gz.suffix.lower() == ".7z":
            _unpack_7z(_gz, DATA_DIR)
        else:
            with zipfile.ZipFile(_gz) as _zf:
                if _zf.testzip() is not None:
                    raise zipfile.BadZipFile("CRC error")
                _zf.extractall(DATA_DIR)
    except (zipfile.BadZipFile, OSError, subprocess.CalledProcessError) as e:
        stop_run([f"'{_gz.name}' is not a valid ZIP ({e}); re-create it and run again"])
    # nested ZIPs (e.g. Kaggle's own download) are unpacked once, next to themselves
    for _z in [z for z in DATA_DIR.rglob("*.zip") if not _is_general(z.name)]:   # nested ZIPs only
        try:
            zipfile.ZipFile(_z).extractall(_z.with_suffix(""))
        except zipfile.BadZipFile:
            print(f"  ⚠ {_z.name} inside {GENERAL_ZIP} is not a valid ZIP; ignored")
    _upload_time = datetime.now(timezone.utc).isoformat()
    save_json(dict(general_zip=str(_gz), sha256=sha256_file(_gz), bytes=_gz.stat().st_size, unpacked_utc=_upload_time), _marker)
    print(f"  unpacked {_gz} → {DATA_DIR}")
else:
    print(f"  using {GENERAL_ZIP} already unpacked in {DATA_DIR} ({json.loads(_marker.read_text())['unpacked_utc']})")
GZ_INFO = json.loads(_marker.read_text())

class DataFail(Exception):
    pass

def _norm(c):
    return re.sub(r"[^a-z0-9]", "", str(c).lower())

def find_col(cols, *keys, exclude=()):
    for c in cols:
        n = _norm(c)
        if all(k in n for k in keys) and not any(e in n for e in exclude):
            return c
    return None

def read_header(p):
    try:
        return list(pd.read_csv(p, nrows=0).columns)
    except Exception:
        return []

SENTINELS = {2147483647, -2147483648, 9999, -9999, -999, 99999, -99999, 65535}
def clean_sentinels(s):
    s = pd.to_numeric(pd.Series(np.asarray(s)), errors="coerce").astype("float64")
    bad = s.isin(SENTINELS) | (s.abs() > 1e6)
    return s.mask(bad).values, int(bad.sum())

def parse_time(series):
    s = series.astype(str).str.strip()
    out = pd.to_datetime(s, errors="coerce", format="ISO8601")
    if out.isna().mean() > 0.5:
        out = pd.to_datetime(s, errors="coerce", dayfirst=True, format="mixed")
    return out

# ---- discover files and check them against the REQUIRED / OPTIONAL lists (FAIL-FAST DATA INTAKE) ----
FILES = sorted(p for p in DATA_DIR.rglob("*.csv")) if DATA_DIR.exists() else []
if FAULT == "no_data":
    FILES = []
MANIFEST = {**{n: dict(v, required=True) for n, v in REQUIRED_FILES.items()}, **{n: dict(v, required=False) for n, v in OPTIONAL_FILES.items()}}
_by_name = {}
for p in FILES:
    _by_name.setdefault(p.name.lower(), []).append(p)
INV, _seen, CHECK = [], {}, []
def _readable(p, role):
    """Readable CSV with the columns its role needs (an HTML error page or an empty file fails)."""
    with open(p, "rb") as fh:
        head = fh.read(512).lstrip().lower()
    if not head or head.startswith(b"<"):
        return False, "empty file or HTML page, not a CSV"
    cols = read_header(p)
    if len(cols) < 2:
        return False, "fewer than 2 columns"
    need = {"array": ("activepower",), "yulara_array": ("activepower",), "weather": ("globalhorizontal",)}.get(role)
    if need and not find_col(cols, *need):
        return False, f"no column matching {'/'.join(need)}"
    if role == "metadata" and not {"file_key", "rating_kw"} <= set(cols):
        return False, "needs columns file_key, rating_kw"
    return True, ""
for name, v in MANIFEST.items():
    hits = _by_name.get(name.lower(), [])
    if not hits:
        CHECK.append(dict(file=name, required=v["required"], role=v["role"], use=v["use"], status="MISSING", detail=""))
        continue
    p = hits[0]; ok, why = _readable(p, v["role"])
    CHECK.append(dict(file=name, required=v["required"], role=v["role"], use=v["use"], status="OK" if ok else "UNREADABLE", detail=why,
                      path=str(p.relative_to(DATA_DIR)), copies=len(hits)))
CHECK = pd.DataFrame(CHECK)
save_csv(CHECK, DIRS["data_audit"] / "required_files_check.csv", quiet=True)
print(CHECK[["file", "required", "status", "detail"]].to_string(index=False))
_miss_req = CHECK[CHECK.required & (CHECK.status != "OK")]
if len(_miss_req):
    stop_run([f"required file {r.file} is {r.status}" + (f" ({r.detail})" if r.detail else "") + f" — needed for {r.use}" for r in _miss_req.itertuples()],
             CHECK.to_dict("records"))
for r in CHECK[~CHECK.required & (CHECK.status != "OK")].itertuples():
    print(f"  ⚠ optional file {r.file} {r.status}: {r.use} will be NOT EVALUABLE")

def _fingerprint(p):
    """Size + MD5 of the first and last 1 MB: identical copies are detected and ignored."""
    h = hashlib.md5(); n = p.stat().st_size
    with open(p, "rb") as fh:
        h.update(fh.read(1 << 20)); fh.seek(max(0, n - (1 << 20))); h.update(fh.read(1 << 20))
    return f"{n}-{h.hexdigest()}"
for p in FILES:
    rel = str(p.relative_to(DATA_DIR)); v = MANIFEST.get(p.name)
    if v is None:
        v = next((vv for nn, vv in MANIFEST.items() if nn.lower() == p.name.lower()), None)
    fp = _fingerprint(p)
    if fp in _seen:
        INV.append(dict(file=rel, kind="duplicate", bytes=p.stat().st_size, n_cols=0, sha256="", source="", note=f"identical to {_seen[fp]}; ignored")); continue
    _seen[fp] = rel
    if v is None:
        INV.append(dict(file=rel, kind="unlisted", bytes=p.stat().st_size, n_cols=0, sha256="", source="", note="not in REQUIRED/OPTIONAL lists; ignored")); continue
    if not CHECK[(CHECK.file.str.lower() == p.name.lower()) & (CHECK.status == "OK")].shape[0]:
        INV.append(dict(file=rel, kind="unreadable", bytes=p.stat().st_size, n_cols=0, sha256="", source="", note="failed the readability check")); continue
    fam = "helper" if v["role"] == "metadata" else ("Kaggle" if v["role"].startswith("kaggle") else "DKASC")
    INV.append(dict(file=rel, kind=v["role"], bytes=p.stat().st_size, n_cols=len(read_header(p)), sha256=sha256_file(p), source=fam, note=v["use"]))
INV = pd.DataFrame(INV, columns=["file", "kind", "bytes", "n_cols", "sha256", "source", "note"])
save_csv(INV, DIRS["data_audit"] / "file_inventory.csv")
print(INV.groupby("kind").size().to_string() if len(INV) else "  no CSV files found")

def load_series(path, maxdate, cols_wanted, start=None):
    """Load timestamp + requested columns, clean sentinels, keep [PHYS_START, maxdate], resample to 15-min means."""
    cols = read_header(path)
    tcol = find_col(cols, "timestamp") or find_col(cols, "datetime") or find_col(cols, "time") or find_col(cols, "date")
    if tcol is None:
        raise DataFail(f"{Path(path).name}: no timestamp column")
    use = {k: find_col(cols, *v[0], exclude=v[1]) for k, v in cols_wanted.items()}
    df = pd.read_csv(path, usecols=[tcol] + [c for c in use.values() if c], low_memory=False, on_bad_lines="skip")   # malformed rows skipped
    ts = parse_time(df[tcol])
    out = pd.DataFrame(index=ts)
    n_sent = {}
    for k, c in use.items():
        if c is not None:
            out[k], n_sent[k] = clean_sentinels(df[c].values)
    # physical-range rules (after sentinel removal): sensor fault codes such as -40 °C are removed
    RANGES = {"temp": (-10.0, 55.0), "ghi": (0.0, 1500.0), "dhi": (0.0, 1000.0), "wind": (0.0, 60.0), "P": (-1.0, None)}
    for k_, (lo, hi) in RANGES.items():
        if k_ in out:
            bad_r = (out[k_] < lo) | ((out[k_] > hi) if hi is not None else False)
            n_sent[k_] = n_sent.get(k_, 0) + int(bad_r.sum()); out[k_] = out[k_].mask(bad_r)
    if "ghi" in out and "dhi" in out:
        bad_r = out["dhi"] > out["ghi"] + 5; n_sent["dhi"] = n_sent.get("dhi", 0) + int(bad_r.sum()); out["dhi"] = out["dhi"].mask(bad_r)
    out = out[out.index.notna()].sort_index()
    out = out[~out.index.duplicated(keep="first")]
    out = out.loc[(out.index >= pd.Timestamp(start or PHYS_START)) & (out.index <= pd.Timestamp(maxdate))]
    if len(out) == 0:
        return out, n_sent, None
    step = pd.Series(out.index).diff().median()
    minc = max(1, int(round(pd.Timedelta("15min") / step)) - 1) if pd.notna(step) and step < pd.Timedelta("15min") else 1
    out = out.resample("15min").mean().where(out.resample("15min").count() >= minc)
    return out, n_sent, step

WANT_ARRAY = {"P": (("activepower",), ()), "ghi": (("globalhorizontal",), ()), "dhi": (("diffusehorizontal",), ()),
              "temp": (("temperature",), ("module", "cell", "panel")), "wind": (("windspeed",), ())}
WANT_WX = {k: v for k, v in WANT_ARRAY.items() if k != "P"}
MAXDATE = CFG["max_date"]

ARR, VALID_ROWS = {}, []
for _, r in INV[INV.kind == "array"].iterrows():
    key = Path(r.file).stem
    try:
        s, ns, step = load_series(DATA_DIR / r.file, MAXDATE, WANT_ARRAY)
        if len(s) == 0 or s["P"].notna().sum() == 0:
            raise DataFail("no power data in the date window")
        ARR[key] = s
        VALID_ROWS.append(dict(array=key, rows_15min=len(s), first=s.index.min(), last=s.index.max(),
                               native_step=str(step), invalid_values_removed=sum(ns.values()), status="LOADED"))
    except Exception as e:
        VALID_ROWS.append(dict(array=key, status=f"REJECTED: {e}"))
WX, wx_source = None, "none"
for _, r in INV[INV.kind == "weather"].iterrows():
    try:
        WX, _, _ = load_series(DATA_DIR / r.file, MAXDATE, WANT_WX); wx_source = r.file; break
    except Exception as e:
        VALID_ROWS.append(dict(array=f"weather:{r.file}", status=f"REJECTED: {e}"))
if FAULT == "no_weather":
    WX, wx_source = None, "none"
if (WX is None or WX.get("ghi") is None or WX["ghi"].notna().sum() == 0) and ARR:
    # fallback: weather columns inside the array files (DKASC array files carry station weather)
    best = max(ARR, key=lambda k: ARR[k]["ghi"].notna().sum() if "ghi" in ARR[k] else 0)
    if "ghi" in ARR[best] and ARR[best]["ghi"].notna().sum() > 0:
        WX = ARR[best][[c for c in ["ghi", "dhi", "temp", "wind"] if c in ARR[best]]].copy(); wx_source = f"array file {best} (fallback)"
print(f"  arrays loaded: {len(ARR)} | weather source: {wx_source}")
if FAULT == "short_data":
    ARR = {k: v.iloc[: 96 * 3] for k, v in ARR.items()}
LOC = pvlib.location.Location(SITE["lat"], SITE["lon"], tz=SITE["tz"], altitude=SITE["alt"])
L_MAX = L_MAX_DESIGN; TEST = pd.Timedelta(days=CFG["test_days"])
META, ALIGN, WX_FLAGS, TARGETS, SOURCE = {}, [], [], [], None
SEL = pd.DataFrame(columns=["array", "target_window_coverage", "source_valid_days", "role"])
if ARR:
    # ---- array metadata ----
    META = {}
    _mp = [DATA_DIR / r.file for _, r in INV[INV.kind == "metadata"].iterrows()]
    if _mp:
        _md = pd.read_csv(_mp[0])
        for _, r in _md.iterrows():
            for k in ARR:
                if str(r["file_key"]).lower() in k.lower():
                    META[k] = dict(rating_kw=float(r["rating_kw"]), tilt=float(r.get("tilt_deg", DEFAULT_TILT)),
                                   azimuth=float(r.get("azimuth_deg", DEFAULT_AZIMUTH)), tech=str(r.get("technology", "")), source="metadata")
    for k, s in ARR.items():
        p = s["P"]; p = p.mask(p < 0, 0.0)          # small negative night readings → 0
        pre = p.loc[: pd.Timestamp(CFG["origins"][0]) - pd.Timedelta("1min")].dropna()
        if k not in META:
            rating = float(np.nanpercentile(pre, 99.9) / 0.95) if len(pre) > 96 else float("nan")
            META[k] = dict(rating_kw=rating, tilt=DEFAULT_TILT, azimuth=DEFAULT_AZIMUTH, tech="", source="estimated: p99.9/0.95 before first origin")
        rating = META[k]["rating_kw"]
        if np.isfinite(rating) and rating > 0:
            n_rel = int((p > 1.3 * rating).sum())
            p = p.mask(p > 1.3 * rating)             # relative rule AFTER sentinel removal (Binding Block 6 A.6)
            META[k]["relative_removed"] = n_rel
        ARR[k]["P"] = p

    # ---- solar geometry, clear-sky power, timestamp alignment ----
    LOC = pvlib.location.Location(SITE["lat"], SITE["lon"], tz=SITE["tz"], altitude=SITE["alt"])
    def clearsky_frame(index, tilt, azimuth, loc=LOC):
        """Clear-sky POA and solar zenith at interval centres; index is naive local standard time (interval start)."""
        centre = (index + pd.Timedelta("7.5min")).tz_localize(loc.tz, nonexistent="shift_forward", ambiguous="NaT")
        sp = loc.get_solarposition(centre)
        cs = loc.get_clearsky(centre, model="simplified_solis")
        poa = pvlib.irradiance.get_total_irradiance(tilt, azimuth, sp["apparent_zenith"], sp["azimuth"],
                                                    cs["dni"], cs["ghi"], cs["dhi"])["poa_global"]
        return pd.DataFrame({"zenith": sp["apparent_zenith"].values, "poa_cs": poa.fillna(0).clip(lower=0).values,
                             "dni_cs": cs["dni"].values}, index=index)

    ALIGN = []; _first = next(iter(ARR))
    for k in list(ARR):
        s = ARR[k]; m = META[k]
        cs = clearsky_frame(s.index, m["tilt"], m["azimuth"])
        s["zenith"], s["poa_cs"] = cs["zenith"].values, cs["poa_cs"].values
        win = s.loc[: pd.Timestamp(CFG["origins"][0]) - pd.Timedelta("1min")]
        prof_p = win.groupby(win.index.hour * 4 + win.index.minute // 15)["P"].mean()
        prof_c = win.groupby(win.index.hour * 4 + win.index.minute // 15)["poa_cs"].mean()
        best, corr = 0, -1
        for lag in range(-8, 9):
            c = np.corrcoef(prof_p.reindex(range(96)).fillna(0).values, np.roll(prof_c.reindex(range(96)).fillna(0).values, lag))[0, 1]
            if lag == 0:
                corr0 = c
            if c > corr:
                best, corr = lag, c
        if best != 0 and corr - corr0 < 0.002:      # pre-declared: shift only for a clear gain (≥ 0.002) over no shift
            best, corr = 0, corr0
        if FAULT == "time_offset" and k == _first:
            best = 12
        status = "OK"
        if best != 0 and abs(best) <= 6:
            s.index = s.index - pd.Timedelta(minutes=15 * best)       # move power to match solar time
            cs = clearsky_frame(s.index, m["tilt"], m["azimuth"]); s["zenith"], s["poa_cs"] = cs["zenith"].values, cs["poa_cs"].values
            status = f"SHIFTED {-15*best:+d} min"
        elif abs(best) > 6:
            status = "REJECTED: timestamp offset > 90 min"
        peak_p = int(prof_p.idxmax()) / 4 if prof_p.notna().any() else np.nan
        peak_c = int(prof_c.idxmax()) / 4 if prof_c.notna().any() else np.nan
        ALIGN.append(dict(array=k, best_lag_steps=best, corr=round(corr, 5), corr_no_shift=round(corr0, 5), peak_hour_power=peak_p, peak_hour_clearsky=peak_c, status=status))
        if status.startswith("REJECTED"):
            ARR.pop(k)
        else:
            rating = m["rating_kw"]
            s["Pcs"] = rating * s["poa_cs"] / 1000.0 * 0.95
            s["day"] = (s["zenith"] < 85) & (s["Pcs"] > 0.05 * rating)
            s["k"] = (s["P"] / s["Pcs"]).where(s["day"]).clip(0, 1.5)
    save_csv(pd.DataFrame(ALIGN), DIRS["data_audit"] / "timestamp_alignment.csv")

    # ---- dead / standby array days (v2.0.4): a day is set to missing when an array's daytime clear-sky index stays
    # below STANDBY_K while the other arrays at the same site produce normally (median daily index above PEER_K).
    # Lesson (STANDARD v2.0.3): 67-Site was offline May 2022 - Jan 2023 at about -0.06 kW and was counted as valid.
    STANDBY_K, PEER_K = 0.05, 0.30
    if FAULT == "dead_array":                       # fault test: first array in standby across the first origin's target window
        _fa = ARR[list(ARR)[0]]; _o = pd.Timestamp(CFG["origins"][0])
        _w = (_fa.index > _o - pd.Timedelta(days=60)) & (_fa.index < _o + pd.Timedelta(days=CFG["test_days"]))
        _fa.loc[_w, "P"] = -0.06; _fa["k"] = (_fa["P"] / _fa["Pcs"]).where(_fa["day"]).clip(0, 1.5)
    _dk = {}
    for k, s in ARR.items():
        bright = s["day"] & (s["Pcs"] > 0.30 * META[k]["rating_kw"])
        _dk[k] = s.loc[bright, "k"].groupby(s.index[bright].normalize()).mean()
    DKD = pd.DataFrame(_dk)
    STANDBY = []
    for k in list(ARR):
        peers = DKD.drop(columns=[k]).median(axis=1) if DKD.shape[1] > 1 else pd.Series(1.0, index=DKD.index)
        dead = DKD.index[(DKD[k] < STANDBY_K) & (peers > PEER_K)]
        if len(dead):
            s = ARR[k]; mask = s.index.normalize().isin(dead)
            s.loc[mask, ["P", "k"]] = np.nan
            months = pd.Series(1, index=dead).groupby(dead.to_period("M")).sum()
            for mth, n in months.items():
                STANDBY.append(dict(array=k, month=str(mth), standby_days=int(n)))
    STANDBY = pd.DataFrame(STANDBY, columns=["array", "month", "standby_days"])
    save_csv(STANDBY, DIRS["data_audit"] / "standby_days.csv", quiet=True)
    if len(STANDBY):
        print("  standby / dead-array days set to missing: " + "; ".join(f"{a}: {int(g.standby_days.sum())} d ({g.month.min()} to {g.month.max()})" for a, g in STANDBY.groupby("array")))
    else:
        print("  standby / dead-array days: none")

    # ---- weather checks (E17: constant / duplicated columns) ----
    WX_FLAGS = []
    if WX is not None:
        for c in WX.columns:
            v = WX[c].dropna()
            WX_FLAGS.append(dict(column=c, n=len(v), std=float(v.std()) if len(v) else np.nan, flag="CONSTANT" if len(v) and v.std() < 1e-9 else ""))
        num = WX.dropna()
        for a in WX.columns:
            for b in WX.columns:
                if a < b and len(num) > 100 and num[a].std() > 0 and num[b].std() > 0 and abs(num[a].corr(num[b])) > 0.999 and a != "ghi":
                    WX_FLAGS.append(dict(column=f"{a}~{b}", n=len(num), std=np.nan, flag="NEAR-IDENTICAL (|r|>0.999); both excluded"))
        for f in WX_FLAGS:
            for c in re.split("~", f["column"]):
                if f["flag"] and c in WX:
                    WX[c] = np.nan
    save_csv(pd.DataFrame(WX_FLAGS, columns=["column", "n", "std", "flag"]), DIRS["data_audit"] / "weather_checks.csv")

    # ---- coverage and pre-declared array selection ----
    L_MAX = L_MAX_DESIGN; TEST = pd.Timedelta(days=CFG["test_days"])
    def coverage(s, a, b):
        w = s.loc[a:b]; d = w["day"]
        return float(w.loc[d, "P"].notna().mean()) if d.sum() else 0.0
    SEL = []
    for k, s in ARR.items():
        covs = [coverage(s, pd.Timestamp(o) - pd.Timedelta(days=L_MAX), pd.Timestamp(o) + TEST - pd.Timedelta("1min")) for o in CFG["origins"]]
        pre = s.loc[:SOURCE_END]
        SEL.append(dict(array=k, rating_kw=round(META[k]["rating_kw"], 2), rating_source=META[k]["source"], tech=META[k]["tech"],
                        target_window_coverage=round(min(covs), 4), source_valid_days=int(pre.loc[pre["day"], "P"].notna().sum() / 48)))
    SEL = pd.DataFrame(SEL, columns=["array", "rating_kw", "rating_source", "tech", "target_window_coverage", "source_valid_days"]).sort_values(["target_window_coverage", "array"], ascending=[False, True])
    thr = 0.95
    TARGETS = list(SEL[SEL.target_window_coverage >= thr].array)[: N_TARGETS_DESIGN]
    if len(TARGETS) < N_TARGETS_DESIGN:
        extra = list(SEL[(SEL.target_window_coverage >= 0.80) & ~SEL.array.isin(TARGETS)].array)[: N_TARGETS_DESIGN - len(TARGETS)]
        TARGETS += extra
    rest = SEL[~SEL.array.isin(TARGETS)].sort_values(["source_valid_days", "array"], ascending=[False, True])
    SOURCE = rest.array.iloc[0] if len(rest) and rest.source_valid_days.iloc[0] >= 180 else None
    TARGETS = TARGETS[: CFG["n_targets"]]          # a reproduction run keeps only the first of the design targets
    SEL["role"] = SEL.array.map(lambda a: "target" if a in TARGETS else ("source" if a == SOURCE else "unused"))
    save_csv(SEL, DIRS["data_audit"] / "array_selection.csv")
    DV = pd.DataFrame(VALID_ROWS)
    DV["role"] = DV["array"].map(lambda a: "target" if a in TARGETS else ("source" if a == SOURCE else ""))
    save_csv(DV, DIRS["data_audit"] / "data_validation.csv")

else:
    save_csv(pd.DataFrame(VALID_ROWS), DIRS['data_audit'] / 'data_validation.csv')

problems = [] if ARR else ["no usable DKASC array files in DATA_DIR (upload the files in the data request)"]
if len(TARGETS) < CFG["n_targets"]:
    problems.append(f"only {len(TARGETS)} target arrays reach ≥80% daytime coverage over {L_MAX} d history + {CFG['test_days']} d test (need {CFG['n_targets']})")
for k in TARGETS:
    r = META[k]["rating_kw"]
    if not (np.isfinite(r) and r > 0):
        problems.append(f"{k}: rating not positive/finite")
    pre = ARR[k].loc[:SOURCE_END, "P"].dropna()
    if len(pre) > 96 and np.isfinite(r) and np.nanpercentile(pre, 99.9) > 1.25 * r:
        problems.append(f"{k}: scale check failed (p99.9 {np.nanpercentile(pre, 99.9):.1f} kW > 1.25 × rating)")
if SOURCE is None:
    problems.append("no source array with ≥180 valid days before 2022 (transfer arm A5 cannot run)")
if WX is None or "ghi" not in WX or WX["ghi"].notna().sum() < 96 * 30:
    problems.append("no weather irradiance (physics arm A4 cannot run)")

_used = INV[~INV.kind.isin(["duplicate", "unlisted", "unreadable"])]
DATA_PROVENANCE = dict(general_zip=GZ_INFO, files=INV.to_dict("records"), sources=DATASET_SOURCES, weather_source=wx_source, targets=TARGETS, source=SOURCE,
                       meta={k: META[k] for k in TARGETS + ([SOURCE] if SOURCE else [])}, max_date_read=MAXDATE,
                       data_hash=hashlib.sha256(json.dumps(sorted((r["file"], r["sha256"]) for r in _used.to_dict("records"))).encode()).hexdigest()[:12])
save_json(DATA_PROVENANCE, DIRS["data_audit"] / "DATA_PROVENANCE.json")
DATA_HASH = DATA_PROVENANCE["data_hash"]
print("\n=== EARLY-OUTPUT CHECK: send these lines to Claude before the long part runs ===")
print(SEL.to_string(index=False))
print("timestamp alignment:", "; ".join(f"{a['array']}: {a['status']}" for a in ALIGN))
print("targets:", TARGETS, "| source:", SOURCE, "| weather:", wx_source)
RUN_ID = f"{MODE}-v{NOTEBOOK_VERSION}-{CFG_HASH}-{DATA_HASH}" + ("-REPRO" if REPRO else "") + ("-REPLICATION" if REPLICATION_ONLY else "")
print("RUN ID:", RUN_ID)
if problems:
    print("DATA VALIDATION FAILED:\n  - " + "\n  - ".join(problems))
    stop_run(["DATA VALIDATION FAILED: " + p_ for p_ in problems])
print("DATA VALIDATION PASSED")
DATA_OK = not problems
RUN_MODELS = len(TARGETS) > 0 and not REPLICATION_ONLY
log_runtime("data", time.time() - t_data)


In [ ]:
# ===== CELL 5 — FEATURES, SPLITS, PHYSICS SIMULATION, LEAKAGE ASSERTIONS =====
def time_feats(idx):
    """Time of day only. Day-of-year features are NOT used: with 14 days of history they cannot generalise to a
    90-day test window in another season (they extrapolate); seasonal sun geometry enters through clear-sky power."""
    tod = (idx.hour * 60 + idx.minute + 7.5) / 1440.0
    return np.c_[np.sin(2 * np.pi * tod), np.cos(2 * np.pi * tod)]

def frame_for(s, rating):
    """Issue-time frame: lags of the clear-sky index (night = 0), targets k(t+h), statics at target times."""
    k_in = s["k"].where(s["day"], 0.0)                     # night inputs are 0 (known), missing daytime stays NaN
    F = pd.DataFrame(index=s.index)
    for j in range(LAGS):
        F[f"lag{j}"] = k_in.shift(j)
    for h in HORIZONS:
        F[f"y{h}"] = s["k"].shift(-h)
        F[f"day{h}"] = s["day"].shift(-h).fillna(False).astype(bool)
        F[f"pcs{h}"] = s["Pcs"].shift(-h)
        F[f"p{h}"] = s["P"].shift(-h)
        tf = time_feats(s.index + pd.Timedelta(minutes=15 * h))
        for i, nm in enumerate(["stod", "ctod"]):
            F[f"{nm}{h}"] = tf[:, i]
        F[f"pcsf{h}"] = F[f"pcs{h}"] / rating
    return F

STAT_COLS = [f"{nm}{h}" for h in HORIZONS for nm in ["stod", "ctod", "pcsf"]]
LAG_COLS = [f"lag{j}" for j in range(LAGS)]

def make_set(F, a, b, rating):
    """Windows whose issue time is in [a, b); a target is kept only if its time is also < b (no peeking past the split)."""
    W = F.loc[(F.index >= a) & (F.index < b)]
    W = W[W[LAG_COLS].notna().all(axis=1)]
    Y, M, PCS, P = [], [], [], []
    for h in HORIZONS:
        ok = W[f"day{h}"].values & W[f"y{h}"].notna().values & ((W.index + pd.Timedelta(minutes=15 * h)) < b)
        Y.append(np.where(ok, W[f"y{h}"].fillna(0).values, 0.0)); M.append(ok)
        PCS.append(W[f"pcs{h}"].fillna(0).values); P.append(W[f"p{h}"].values)
    M = np.stack(M, 1); keep = M.any(1)
    return dict(seq=W[LAG_COLS].values[keep].astype("float32"), stat=W[STAT_COLS].values[keep].astype("float32"),
                y=np.stack(Y, 1)[keep].astype("float32"), m=M[keep], pcs=np.stack(PCS, 1)[keep].astype("float32"),
                p=np.stack(P, 1)[keep].astype("float32"), t=W.index[keep], rating=rating)

def subset(D, idx):
    return {k: (v[idx] if isinstance(v, np.ndarray) else (v[idx] if isinstance(v, pd.DatetimeIndex) else v)) for k, v in D.items()}

def concat_sets(*Ds):
    out = {}
    for k in Ds[0]:
        if isinstance(Ds[0][k], np.ndarray):
            out[k] = np.concatenate([D[k] for D in Ds])
        elif isinstance(Ds[0][k], pd.DatetimeIndex):
            out[k] = Ds[0][k].append([D[k] for D in Ds[1:]])
        else:
            out[k] = Ds[0][k]
    return out

def simulate_physics(target, a, b):
    """pvlib simulation of the target array's power from weather data in [a, b) — weather only, no target power."""
    if WX is None or "ghi" not in WX:
        return None
    m = META[target]; w = WX.loc[(WX.index >= a) & (WX.index < b)].copy()
    w = w[w["ghi"].notna()]
    if len(w) < 96 * 30:
        return None
    centre = (w.index + pd.Timedelta("7.5min")).tz_localize(LOC.tz, nonexistent="shift_forward", ambiguous="NaT")
    sp = LOC.get_solarposition(centre)
    ghi = w["ghi"].clip(lower=0).values
    if "dhi" in w and w["dhi"].notna().mean() > 0.8:
        dhi = w["dhi"].clip(lower=0).fillna(0).values
        dni = ((ghi - dhi) / np.cos(np.radians(sp["apparent_zenith"].clip(upper=89).values))).clip(min=0)
    else:
        e = pvlib.irradiance.erbs(ghi, sp["apparent_zenith"].values, centre)
        dni, dhi = np.nan_to_num(np.asarray(e["dni"], dtype=float)), np.nan_to_num(np.asarray(e["dhi"], dtype=float))
    poa = pvlib.irradiance.get_total_irradiance(m["tilt"], m["azimuth"], sp["apparent_zenith"].values, sp["azimuth"].values,
                                                dni, ghi, dhi)["poa_global"]
    poa = np.nan_to_num(np.asarray(poa, dtype=float)).clip(min=0)
    tamb = w["temp"].fillna(25.0).values if "temp" in w else np.full(len(w), 25.0)
    tc = tamb + poa / 800.0 * 25.0
    psim = np.clip(m["rating_kw"] * poa / 1000.0 * (1 - 0.004 * (tc - 25.0)) * 0.96, 0, m["rating_kw"])
    sim = pd.DataFrame({"P": psim}, index=w.index).reindex(pd.date_range(w.index.min(), w.index.max(), freq="15min"))
    cs = clearsky_frame(sim.index, m["tilt"], m["azimuth"])
    sim["zenith"], sim["poa_cs"] = cs["zenith"].values, cs["poa_cs"].values
    sim["Pcs"] = m["rating_kw"] * sim["poa_cs"] / 1000.0 * 0.95
    sim["day"] = (sim["zenith"] < 85) & (sim["Pcs"] > 0.05 * m["rating_kw"])
    sim["k"] = (sim["P"] / sim["Pcs"]).where(sim["day"]).clip(0, 1.5)
    return sim

def split_bounds(o, L):
    o = pd.Timestamp(o); h0 = o - pd.Timedelta(days=L)
    v0 = h0 + pd.Timedelta(days=round(L * (1 - VAL_FRAC), 6))
    v0 = v0.floor("1D")
    return h0, v0, o, o + TEST

FRAMES = {k: frame_for(ARR[k], META[k]["rating_kw"]) for k in TARGETS + ([SOURCE] if SOURCE else [])} if RUN_MODELS else {}
SPLITS = []
for k in TARGETS:
    for o in CFG["origins"]:
        for L in CFG["lengths"]:
            h0, v0, o_, t1 = split_bounds(o, L)
            SPLITS.append(dict(target=k, origin=o, L=L, train_start=h0, val_start=v0, test_start=o_, test_end=t1))
SPLITS = pd.DataFrame(SPLITS)
if len(SPLITS):
    save_csv(SPLITS, DIRS["data_audit"] / "split_table.csv")
    # automated leakage assertions
    assert (SPLITS.train_start < SPLITS.val_start).all() and (SPLITS.val_start < SPLITS.test_start).all()
    assert (SPLITS.test_end <= pd.Timestamp(MAXDATE) + pd.Timedelta("1min")).all(), "test window beyond the date read for this mode"
    if MODE != "FULL_PAPER_RUN":
        assert (SPLITS.test_end <= pd.Timestamp("2023-01-01")).all(), "confirmatory 2023 data must not enter SMOKE/STANDARD"
    assert pd.Timestamp(SOURCE_END) < pd.Timestamp(min(CFG["origins"])), "transfer source must end before every origin"
    print(SPLITS.to_string(index=False))


In [ ]:
# ===== CELL 6 — MODELS: LSTM (equal-update training), LightGBM (equal trials), DDPM, simple augmenters =====
class LSTMNet(nn.Module):
    """Base-paper-sized LSTM (32 units → Dense 8 ReLU → Dropout 0.2) with target-time static features in the head."""
    def __init__(self, n_stat, n_out=len(HORIZONS), hid=32):
        super().__init__()
        self.lstm = nn.LSTM(1, hid, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hid + n_stat, 8), nn.ReLU(), nn.Dropout(0.2), nn.Linear(8, n_out))
    def forward(self, seq, stat):
        _, (h, _) = self.lstm(torch.flip(seq, [1]).unsqueeze(-1))     # oldest → newest
        return self.head(torch.cat([h[-1], stat[:, NN_STAT_IDX]], 1))

def _t(x):
    return torch.as_tensor(x, dtype=torch.float32, device=DEVICE)

PCSF_IDX = [STAT_COLS.index(f"pcsf{h}") for h in HORIZONS]
NN_STAT_IDX = [i for i, c in enumerate(STAT_COLS) if c.endswith(str(HORIZONS[0]))]   # LSTM/DDPM: one set of statics (no collinear copies)
def masked_mse(pred, y, m, w=None):
    """MSE on the clear-sky index, weighted by (clear-sky power / rating)^2 so training matches the kW evaluation."""
    m = m.float() * (w if w is not None else 1.0)
    return ((pred - y) ** 2 * m).sum() / m.sum().clamp(min=1e-9)

def kw_weight(stat):
    return torch.as_tensor(np.asarray(stat)[:, PCSF_IDX] ** 2, dtype=torch.float32, device=DEVICE)

def eval_loss(net, D):
    net.eval()
    with torch.no_grad():
        out = []
        for i in range(0, len(D["y"]), 4096):
            sl = slice(i, i + 4096)
            w = kw_weight(D["stat"][sl]); mm = torch.as_tensor(D["m"][sl], device=DEVICE)
            out.append((((net(_t(D["seq"][sl]), _t(D["stat"][sl])) - _t(D["y"][sl])) ** 2) * mm.float() * w).sum().item())
            den = (mm.float() * w).sum().item(); out[-1] = (out[-1], den)
    num = sum(a for a, _ in out); den = sum(b for _, b in out)
    return num / max(den, 1e-9)

def train_lstm(Dtr, Dva, updates, seed, init_state=None, lr=1e-3):
    """Exactly `updates` optimizer steps (batches drawn with replacement), best weights by real-data validation loss."""
    set_seed(seed)
    net = LSTMNet(len(NN_STAT_IDX)).to(DEVICE)
    if init_state is not None:
        net.load_state_dict(init_state)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    n = len(Dtr["y"]); rng = np.random.RandomState(seed)
    every = max(1, updates // 20); best, best_state = float("inf"), None
    seq, stat, y, m = _t(Dtr["seq"]), _t(Dtr["stat"]), _t(Dtr["y"]), torch.as_tensor(Dtr["m"], device=DEVICE)
    W = kw_weight(Dtr["stat"])
    for u in range(1, updates + 1):
        net.train(); b = torch.as_tensor(rng.randint(0, n, BATCH), device=DEVICE)
        loss = masked_mse(net(seq[b], stat[b]), y[b], m[b], W[b])
        opt.zero_grad(); loss.backward(); opt.step()
        if u % every == 0 or u == updates:
            vl = eval_loss(net, Dva)
            if vl < best:
                best, best_state = vl, {k: v.detach().clone() for k, v in net.state_dict().items()}
    net.load_state_dict(best_state)
    if FAULT == "nonfinite_pred":
        with torch.no_grad():
            net.head[-1].bias.fill_(float("nan"))
    return net, best

def predict_lstm(net, D):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(D["y"]), 4096):
            sl = slice(i, i + 4096); out.append(net(_t(D["seq"][sl]), _t(D["stat"][sl])).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(HORIZONS)), "float32")

def pretrain_lstm(D, epochs, seed):
    n = len(D["y"]); cut = int(n * 0.9)
    upd = max(1, epochs * math.ceil(cut / BATCH))
    net, _ = train_lstm(subset(D, slice(0, cut)), subset(D, slice(cut, n)), upd, seed)
    return {k: v.detach().clone() for k, v in net.state_dict().items()}

# ---- LightGBM: one model per horizon, equal random-search trials, early stopping on real validation only ----
def lgb_X(D, hi):
    h = HORIZONS[hi]; sc = [i for i, c in enumerate(STAT_COLS) if c.endswith(str(h))]
    return np.c_[D["seq"], D["stat"][:, sc]]

def lgb_fit(Dtr, Dva, hi, trials, rounds, seed, init=None):
    mtr, mva = Dtr["m"][:, hi], Dva["m"][:, hi]
    Xtr, ytr, Xva, yva = lgb_X(Dtr, hi)[mtr], Dtr["y"][mtr, hi], lgb_X(Dva, hi)[mva], Dva["y"][mva, hi]
    wtr, wva = Dtr["stat"][mtr, PCSF_IDX[hi]] ** 2, Dva["stat"][mva, PCSF_IDX[hi]] ** 2      # same kW-matched weights
    if len(ytr) < 20 or len(yva) < 5:
        raise ValueError(f"too few rows for LightGBM (train {len(ytr)}, val {len(yva)})")
    rng = np.random.RandomState(seed); best = (float("inf"), None, None)
    for t in range(trials):
        p = dict(objective="regression", learning_rate=0.05, num_leaves=15, min_data_in_leaf=20, feature_fraction=0.9,
                 bagging_fraction=0.9, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=seed * 100 + t, num_threads=min(4, os.cpu_count() or 1))
        if t > 0:
            p.update(learning_rate=float(rng.choice([0.03, 0.05, 0.1])), num_leaves=int(rng.choice([7, 15, 31, 63])),
                     min_data_in_leaf=int(rng.choice([5, 10, 20, 40])), feature_fraction=float(rng.choice([0.7, 0.85, 1.0])),
                     lambda_l2=float(rng.choice([0.0, 1.0, 5.0])))
        bst = lgb.train(p, lgb.Dataset(Xtr, ytr, weight=wtr), rounds, valid_sets=[lgb.Dataset(Xva, yva, weight=wva)],
                        callbacks=[lgb.early_stopping(50, verbose=False)], init_model=init)
        sc = bst.best_score["valid_0"]["l2"]
        if sc < best[0]:
            best = (sc, bst, p)
    return best[1], best[2]

def lgb_pretrain(D, hi, seed, rounds):
    n = len(D["y"]); cut = int(n * 0.9)
    bst, _ = lgb_fit(subset(D, slice(0, cut)), subset(D, slice(cut, n)), hi, 1, min(rounds, 500), seed)
    return bst

# ---- simple augmenters (training rows only) ----
def aug_replicate(D, seed, frac=None):
    frac = AUG_FRAC if frac is None else frac
    rng = np.random.RandomState(seed); idx = rng.choice(len(D["y"]), int(frac * len(D["y"])), replace=True)
    return concat_sets(D, subset(D, idx)), 0

def aug_jitter(D, seed, frac=None):
    frac = AUG_FRAC if frac is None else frac
    rng = np.random.RandomState(seed); idx = rng.choice(len(D["y"]), int(frac * len(D["y"])), replace=True)
    E = subset(D, idx); E = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in E.items()}
    sd_seq = D["seq"].std(0) + 1e-6; sd_y = D["y"][D["m"].all(1)].std(0) + 1e-6 if D["m"].all(1).any() else np.ones(len(HORIZONS))
    E["seq"] = np.clip(E["seq"] + rng.normal(0, JITTER_SIGMA, E["seq"].shape) * sd_seq, 0, 1.5).astype("float32")
    E["y"] = np.clip(E["y"] + rng.normal(0, JITTER_SIGMA, E["y"].shape) * sd_y, 0, 1.5).astype("float32") * E["m"]
    return concat_sets(D, E), 0

# ---- DDPM (base-paper settings: T=200, β ∈ [1e-4, 0.02] linear, MLP 128-128, time embedding 64), conditional on time features ----
class EpsNet(nn.Module):
    def __init__(self, dx, dc, temb=64):
        super().__init__(); self.temb = temb
        self.net = nn.Sequential(nn.Linear(dx + dc + temb, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, dx))
    def forward(self, x, c, t):
        half = self.temb // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, device=x.device) / half)
        e = torch.cat([torch.sin(t[:, None] * f), torch.cos(t[:, None] * f)], 1)
        return self.net(torch.cat([x, c, e], 1))

DDPM_T = 200
_betas = torch.linspace(1e-4, 0.02, DDPM_T)
_alphas = 1 - _betas
_abar = torch.cumprod(_alphas, 0)

def train_sample_ddpm(D, n_new, iters, seed):
    """Train on real training windows of the target only; return n_new synthetic windows (clipped to physical k range)."""
    set_seed(seed)
    full = D["m"].all(1)
    if full.sum() < 32:
        raise ValueError(f"too few complete training windows for DDPM ({int(full.sum())})")
    X = np.c_[D["seq"][full], D["y"][full]].astype("float32"); C = D["stat"][full][:, NN_STAT_IDX] if D["stat"].shape[1] else D["stat"][full]
    Cfull = D["stat"][full]
    mu, sd = X.mean(0), X.std(0) + 1e-3
    Xn = _t((X - mu) / sd); Ct = _t(C)
    net = EpsNet(X.shape[1], C.shape[1]).to(DEVICE); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    ab = _abar.to(DEVICE); rng = np.random.RandomState(seed)
    for it in range(iters):
        b = torch.as_tensor(rng.randint(0, len(Xn), 64), device=DEVICE)
        t = torch.randint(0, DDPM_T, (len(b),), device=DEVICE)
        eps = torch.randn_like(Xn[b]); a = ab[t][:, None]
        loss = ((net(a.sqrt() * Xn[b] + (1 - a).sqrt() * eps, Ct[b], t.float()) - eps) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    if FAULT == "ddpm_fail":
        raise RuntimeError("injected DDPM failure")
    net.eval(); ci = rng.randint(0, len(C), n_new); c = Ct[torch.as_tensor(ci, device=DEVICE)]
    x = torch.randn(n_new, X.shape[1], device=DEVICE)
    with torch.no_grad():
        for t in range(DDPM_T - 1, -1, -1):
            tt = torch.full((n_new,), float(t), device=DEVICE)
            eh = net(x, c, tt); a_t, ab_t, b_t = _alphas[t].item(), _abar[t].item(), _betas[t].item()
            x = (x - b_t / math.sqrt(1 - ab_t) * eh) / math.sqrt(a_t)
            if t > 0:
                x = x + math.sqrt(b_t) * torch.randn_like(x)
    raw = x.cpu().numpy() * sd + mu
    frac_in = float(((raw >= 0) & (raw <= 1.5)).mean())
    syn = np.clip(raw, 0, 1.5).astype("float32")
    if not np.isfinite(syn).all():
        raise ValueError("non-finite DDPM samples")
    E = dict(seq=syn[:, :LAGS], stat=Cfull[ci], y=syn[:, LAGS:], m=np.ones((n_new, len(HORIZONS)), bool),
             pcs=np.zeros((n_new, len(HORIZONS)), "float32"), p=np.full((n_new, len(HORIZONS)), np.nan, "float32"),
             t=pd.DatetimeIndex([pd.NaT] * n_new), rating=D["rating"])
    val = dict(frac_raw_in_range=frac_in, real_mean=float(X.mean()), syn_mean=float(syn.mean()), real_sd=float(X.std()), syn_sd=float(syn.std()),
               w1_y=float(np.mean([sst.wasserstein_distance(X[:, LAGS + i], syn[:, LAGS + i]) for i in range(len(HORIZONS))])))
    return E, val

def n_params(net):
    return int(sum(p.numel() for p in net.parameters()))


In [ ]:
# ===== CELL 7 — TASK LEDGER AND RUNNER (resumable; reuse only on matching version, config and data) =====
def tid_name(*parts):
    return re.sub(r"[^A-Za-z0-9_.=-]", "_", "__".join(str(p) for p in parts))

def save_task(name, pred, D, meta):
    path = DIRS["tasks"] / f"{name}.npz"; tmp = DIRS["tasks"] / f"{name}.tmp.npz"
    meta = dict(meta, cfg_hash=CFG_HASH, data_hash=DATA_HASH, fmt=FORMAT_VERSION, version=NOTEBOOK_VERSION, n=int(len(D["y"])))
    np.savez_compressed(tmp, pred=pred.astype("float32"), y=D["y"], m=D["m"], pcs=D["pcs"], p=D["p"],
                        t=np.asarray(D["t"].values.astype("datetime64[ns]").astype("int64")), meta=np.array(json.dumps(meta)))
    os.replace(tmp, path)

def load_task(name, expect_n=None):
    """Return the task file only if it is complete, matches this run's config/data/version and has finite predictions."""
    path = DIRS["tasks"] / f"{name}.npz"
    if not path.exists():
        return None
    try:
        z = np.load(path, allow_pickle=False); meta = json.loads(str(z["meta"]))
        if meta.get("cfg_hash") != CFG_HASH or meta.get("data_hash") != DATA_HASH or meta.get("fmt") != FORMAT_VERSION:
            return None
        out = {k: z[k] for k in ["pred", "y", "m", "pcs", "p", "t"]}; out["meta"] = meta
        if expect_n is not None and len(out["y"]) != expect_n:
            return None
        if out["pred"].shape != out["y"].shape or not np.isfinite(out["pred"][out["m"]]).all():
            return None
        return out
    except Exception:
        return None

GEN_VALID, COST = [], []
def run_all():
    if not RUN_MODELS:
        if REPLICATION_ONLY:
            restore_checkpoint()               # brings back replication seeds saved before a pause or disconnect
        print("Replication-only run: main benchmark skipped." if REPLICATION_ONLY else "No target arrays: modelling skipped (software gate will FAIL)."); return
    restore_checkpoint()
    combos = [(k, o, L, s) for k in TARGETS for o in CFG["origins"] for L in CFG["lengths"] for s in CFG["seeds"]]
    # register every expected task first
    for k, o, L, s in combos:
        LEDGER.expect(tid_name("SP", "ref", k, o, f"L{L}", "s0"), "reference")
        LEDGER.expect(tid_name("GEN", k, o, f"L{L}", f"s{s}"), "ddpm_generator")
        for arm in ARMS:
            for bb in BACKBONES:
                LEDGER.expect(tid_name(arm, bb, k, o, f"L{L}", f"s{s}"), "model")
    if CFG["oracle"]:
        for k in TARGETS:
            for o in CFG["origins"]:
                for s in CFG["seeds"]:
                    for bb in BACKBONES:
                        LEDGER.expect(tid_name("A6_oracle", bb, k, o, "Lall", f"s{s}"), "oracle")
    if not REPRO:   # sensitivity (pre-declared, exploratory): augmentation amount, shortest length, first origin, first seed
        for k in TARGETS:
            for f in SENS_AUG_FRACS:
                for arm in ["A1_replicate", "A2_jitter", "A3_ddpm"]:
                    for bb in BACKBONES:
                        LEDGER.expect(tid_name(f"SENS-{arm}-f{f:.2f}", bb, k, CFG["origins"][0], f"L{min(CFG['lengths'])}", f"s{CFG['seeds'][0]}"), "sensitivity")
    LEDGER.save()
    n_total = sum(1 for r in LEDGER.rows.values() if r["kind"] in ("model", "oracle", "sensitivity"))
    prog = Progress("models", n_total)
    pre_cache = {}
    def pretrained(kind, k, o, s, bb, hi=None):
        key = (kind, k if kind == "phys" else "src", o if kind == "phys" else "-", s, bb, hi)
        if key in pre_cache:
            return pre_cache[key]
        t0 = time.time()
        if kind == "src":
            if SOURCE is None:
                raise ValueError("no source array")
            D = make_set(FRAMES[SOURCE], pd.Timestamp(PHYS_START), pd.Timestamp(SOURCE_END), META[SOURCE]["rating_kw"])
        else:
            sim = simulate_physics(k, pd.Timestamp(PHYS_START), pd.Timestamp(o) - pd.Timedelta(days=L_MAX_DESIGN))
            if sim is None:
                raise ValueError("physics simulation unavailable (no weather irradiance before the history window)")
            D = make_set(frame_for(sim, META[k]["rating_kw"]), sim.index.min(), sim.index.max(), META[k]["rating_kw"])
        if len(D["y"]) < 200:
            raise ValueError(f"pre-training set too small ({len(D['y'])} windows)")
        obj = pretrain_lstm(D, CFG["pre_epochs"], s) if bb == "lstm" else lgb_pretrain(D, hi, s, CFG["lgbm_rounds"])
        pre_cache[key] = obj
        COST.append(dict(stage=f"pretrain_{kind}", backbone=bb, target=k, origin=o, seed=s, horizon=hi, seconds=round(time.time() - t0, 2), n_windows=len(D["y"])))
        return obj

    for k in TARGETS:
        rating = META[k]["rating_kw"]
        for o in CFG["origins"]:
            for L in CFG["lengths"]:
                h0, v0, o_, t1 = split_bounds(o, L)
                Dtr, Dva, Dte = make_set(FRAMES[k], h0, v0, rating), make_set(FRAMES[k], v0, o_, rating), make_set(FRAMES[k], o_, t1, rating)
                # smart persistence reference (clear-sky index persistence; night issue → training median)
                sp_name = tid_name("SP", "ref", k, o, f"L{L}", "s0")
                if load_task(sp_name, len(Dte["y"])) is not None:
                    LEDGER.done(sp_name, reused=True)
                else:
                    med = float(np.median(Dtr["y"][Dtr["m"]])) if Dtr["m"].any() else 1.0
                    lag0 = Dte["seq"][:, 0]; is_day = Dte["stat"][:, STAT_COLS.index(f"pcsf{HORIZONS[0]}")] > 0.05
                    sp = np.where((lag0 > 0)[:, None], lag0[:, None], med).repeat(len(HORIZONS), 1).astype("float32")
                    save_task(sp_name, sp, Dte, dict(task=sp_name)); LEDGER.done(sp_name)
                for s in CFG["seeds"]:
                    n_real = len(Dtr["y"]); U = max(CFG["e_base"] * math.ceil(max(n_real, 1) / BATCH), CFG["u_min"])
                    gen_name = tid_name("GEN", k, o, f"L{L}", f"s{s}"); gen_path = DIRS["tasks"] / f"{gen_name}.syn.npz"
                    syn = None
                    try:
                        if gen_path.exists():
                            z = np.load(gen_path, allow_pickle=False); gm = json.loads(str(z["meta"]))
                            if gm.get("cfg_hash") == CFG_HASH and gm.get("data_hash") == DATA_HASH and np.isfinite(z["seq"]).all():
                                n_new = len(z["y"])
                                syn = dict(seq=z["seq"], stat=z["stat"], y=z["y"], m=np.ones((n_new, len(HORIZONS)), bool),
                                           pcs=np.zeros((n_new, len(HORIZONS)), "float32"), p=np.full((n_new, len(HORIZONS)), np.nan, "float32"),
                                           t=pd.DatetimeIndex([pd.NaT] * n_new), rating=rating)
                                if "gv" in gm:
                                    GEN_VALID.append(gm["gv"])
                                LEDGER.done(gen_name, reused=True)
                        if syn is None:
                            t0 = time.time(); syn, gv = train_sample_ddpm(Dtr, int(AUG_FRAC * n_real), CFG["ddpm_iters"], s)
                            gs = time.time() - t0
                            gvrow = dict(target=k, origin=o, L=L, seed=s, n_real=n_real, n_syn=len(syn["y"]), seconds=round(gs, 2), **gv)
                            tmp = DIRS["tasks"] / f"{gen_name}.tmp.npz"
                            np.savez_compressed(tmp, seq=syn["seq"], stat=syn["stat"], y=syn["y"],
                                                meta=np.array(json.dumps(dict(cfg_hash=CFG_HASH, data_hash=DATA_HASH, seconds=gs, gv=gvrow))))
                            os.replace(tmp, gen_path)
                            GEN_VALID.append(gvrow)
                            COST.append(dict(stage="ddpm_generator", backbone="-", target=k, origin=o, L=L, seed=s, seconds=round(gs, 2), n_windows=n_real))
                            LEDGER.done(gen_name, seconds=gs)
                    except Exception as e:
                        LEDGER.fail(gen_name, f"{type(e).__name__}: {e}"); syn = None
                    for arm in ARMS:
                        for bb in BACKBONES:
                            name = tid_name(arm, bb, k, o, f"L{L}", f"s{s}")
                            prog.step(name)
                            if FAULT == "missing_arm" and arm == "A5_transfer":
                                LEDGER.fail(name, "injected: arm not run"); continue
                            if load_task(name, len(Dte["y"])) is not None:
                                LEDGER.done(name, reused=True); continue
                            t0 = time.time()
                            try:
                                if arm == "A0_target":
                                    D = Dtr
                                elif arm == "A1_replicate":
                                    D, _ = aug_replicate(Dtr, s)
                                elif arm == "A2_jitter":
                                    D, _ = aug_jitter(Dtr, s)
                                elif arm == "A3_ddpm":
                                    if syn is None:
                                        raise RuntimeError("DDPM generator failed for this split")
                                    D = concat_sets(Dtr, syn)
                                else:
                                    D = Dtr
                                kind = "src" if arm == "A5_transfer" else ("phys" if arm == "A4_physics" else None)
                                if bb == "lstm":
                                    init = pretrained(kind, k, o, s, "lstm") if kind else None
                                    net, vl = train_lstm(D, Dva, U, s, init_state=init, lr=5e-4 if kind else 1e-3)
                                    pred = predict_lstm(net, Dte); npar = n_params(net); hp = dict(updates=U, lr=5e-4 if kind else 1e-3, batch=BATCH)
                                else:
                                    pred = np.zeros_like(Dte["y"]); hp = {}
                                    for hi in range(len(HORIZONS)):
                                        init = pretrained(kind, k, o, s, "lgbm", hi) if kind else None
                                        bst, p = lgb_fit(D, Dva, hi, CFG["lgbm_trials"], CFG["lgbm_rounds"], s, init=init)
                                        pred[:, hi] = bst.predict(lgb_X(Dte, hi), num_iteration=bst.best_iteration)
                                        hp[f"h{HORIZONS[hi]}"] = dict(p, best_iteration=bst.best_iteration)
                                    vl, npar = float("nan"), -1
                                pred = np.clip(pred, 0, 1.5)
                                if not np.isfinite(pred[Dte["m"]]).all():
                                    raise ValueError("non-finite predictions")
                                sec = time.time() - t0
                                save_task(name, pred, Dte, dict(task=name, val_loss=vl, n_params=npar, hp=hp, seconds=sec, n_train=len(D["y"])))
                                COST.append(dict(stage="train", arm=arm, backbone=bb, target=k, origin=o, L=L, seed=s, seconds=round(sec, 2), n_windows=len(D["y"])))
                                LEDGER.done(name, seconds=sec)
                            except Exception as e:
                                LEDGER.fail(name, f"{type(e).__name__}: {e}")
                    LEDGER.save(); checkpoint_zip()
        if not REPRO:
            o = CFG["origins"][0]; L = min(CFG["lengths"]); s = CFG["seeds"][0]
            h0, v0, o_, t1 = split_bounds(o, L)
            Dtr, Dva, Dte = make_set(FRAMES[k], h0, v0, rating), make_set(FRAMES[k], v0, o_, rating), make_set(FRAMES[k], o_, t1, rating)
            n_real = len(Dtr["y"]); U = max(CFG["e_base"] * math.ceil(max(n_real, 1) / BATCH), CFG["u_min"])
            for f in SENS_AUG_FRACS:
                syn_f = None
                for arm in ["A1_replicate", "A2_jitter", "A3_ddpm"]:
                    for bb in BACKBONES:
                        name = tid_name(f"SENS-{arm}-f{f:.2f}", bb, k, o, f"L{L}", f"s{s}"); prog.step(name)
                        if load_task(name, len(Dte["y"])) is not None:
                            LEDGER.done(name, reused=True); continue
                        t0 = time.time()
                        try:
                            if arm == "A1_replicate":
                                D, _ = aug_replicate(Dtr, s, f)
                            elif arm == "A2_jitter":
                                D, _ = aug_jitter(Dtr, s, f)
                            else:
                                if syn_f is None:
                                    syn_f, _ = train_sample_ddpm(Dtr, int(f * n_real), CFG["ddpm_iters"], s)
                                D = concat_sets(Dtr, syn_f)
                            if bb == "lstm":
                                net, vl = train_lstm(D, Dva, U, s); pred = predict_lstm(net, Dte)
                            else:
                                pred = np.zeros_like(Dte["y"])
                                for hi in range(len(HORIZONS)):
                                    bst, _ = lgb_fit(D, Dva, hi, CFG["lgbm_trials"], CFG["lgbm_rounds"], s)
                                    pred[:, hi] = bst.predict(lgb_X(Dte, hi), num_iteration=bst.best_iteration)
                            pred = np.clip(pred, 0, 1.5)
                            if not np.isfinite(pred[Dte["m"]]).all():
                                raise ValueError("non-finite predictions")
                            save_task(name, pred, Dte, dict(task=name, aug_frac=f, seconds=time.time() - t0, n_train=len(D["y"])))
                            LEDGER.done(name, seconds=time.time() - t0)
                        except Exception as e:
                            LEDGER.fail(name, f"{type(e).__name__}: {e}")
            LEDGER.save(); checkpoint_zip()
        if CFG["oracle"]:
            for o in CFG["origins"]:
                Dall = make_set(FRAMES[k], pd.Timestamp(PHYS_START), pd.Timestamp(o), rating)
                cut = int(len(Dall["y"]) * (1 - VAL_FRAC)); Dte = make_set(FRAMES[k], pd.Timestamp(o), pd.Timestamp(o) + TEST, rating)
                for s in CFG["seeds"]:
                    for bb in BACKBONES:
                        name = tid_name("A6_oracle", bb, k, o, "Lall", f"s{s}"); prog.step(name)
                        if load_task(name, len(Dte["y"])) is not None:
                            LEDGER.done(name, reused=True); continue
                        t0 = time.time()
                        try:
                            Dtr, Dva = subset(Dall, slice(0, cut)), subset(Dall, slice(cut, len(Dall["y"])))
                            if bb == "lstm":
                                net, vl = train_lstm(Dtr, Dva, min(max(CFG["e_base"] * math.ceil(cut / BATCH), CFG["u_min"]), 30000), s)
                                pred = predict_lstm(net, Dte)
                            else:
                                pred = np.zeros_like(Dte["y"])
                                for hi in range(len(HORIZONS)):
                                    bst, _ = lgb_fit(Dtr, Dva, hi, CFG["lgbm_trials"], CFG["lgbm_rounds"], s)
                                    pred[:, hi] = bst.predict(lgb_X(Dte, hi), num_iteration=bst.best_iteration)
                            save_task(name, np.clip(pred, 0, 1.5), Dte, dict(task=name, seconds=time.time() - t0, n_train=cut))
                            LEDGER.done(name, seconds=time.time() - t0)
                        except Exception as e:
                            LEDGER.fail(name, f"{type(e).__name__}: {e}")
        LEDGER.save(); checkpoint_zip()

t_run = time.time()
try:
    run_all()
except Exception as e:
    print("⚠ runner stopped:", type(e).__name__, e); traceback.print_exc()
LEDGER.save()
if GEN_VALID:
    save_csv(pd.DataFrame(GEN_VALID), DIRS["tables"] / "synthetic_validity.csv")
if COST:
    save_csv(pd.DataFrame(COST), DIRS["logs"] / "cost_raw.csv")
log_runtime("models", time.time() - t_run, units=len(LEDGER.rows))
print(LEDGER.df().groupby(["kind", "status"]).size().to_string())


In [ ]:
# ===== CELL 8 — EVALUATION AND STATISTICS (seed-mean forecasts, daytime only, kW and capacity-normalised) =====
t_eval = time.time()
RES = {}                                    # (arm, bb, target, origin, L, seed) -> task arrays
for name, r in LEDGER.rows.items():
    if r["status"] != "COMPLETED" or r["kind"] not in ("model", "reference", "oracle"):
        continue
    arm, bb, k, o, Ls, ss = name.split("__")
    k_real = next((t for t in TARGETS if tid_name(t) == k), k)
    z = load_task(name)
    if z is not None:
        RES[(arm, bb, k_real, o, Ls, int(ss[1:]))] = z

def kw_err(z, pred=None):
    pred = z["pred"] if pred is None else pred
    return (pred - z["y"]) * z["pcs"], z["m"]

def metrics_from(e, m, rating):
    e = e[m]
    if e.size == 0:
        return dict(n=0, MAE=np.nan, RMSE=np.nan, MBE=np.nan, nMAE=np.nan, nRMSE=np.nan)
    rmse = float(np.sqrt(np.mean(e ** 2))); mae = float(np.mean(np.abs(e)))
    if FAULT == "wrong_metric":
        rmse *= 1.05                                   # injected fault: the metric self-test must catch this
    return dict(n=int(e.size), MAE=mae, RMSE=rmse, MBE=float(np.mean(e)), nMAE=mae / rating, nRMSE=rmse / rating)

# seed-mean forecasts
SM = {}
for (arm, bb, k, o, Ls, s), z in RES.items():
    SM.setdefault((arm, bb, k, o, Ls), []).append(z)
SMZ = {}
for key, zs in SM.items():
    z = dict(zs[0]); z["pred"] = np.mean([q["pred"] for q in zs], 0); z["n_seeds"] = len(zs); SMZ[key] = z

rows, seed_rows = [], []
for (arm, bb, k, o, Ls), z in SMZ.items():
    rating = META[k]["rating_kw"]; sp = SMZ.get(("SP", "ref", k, o, Ls))
    for hi, h in enumerate(HORIZONS):
        e, m = kw_err(z); met = metrics_from(e[:, hi], m[:, hi], rating)
        if sp is not None and arm != "A6_oracle":
            es, ms = kw_err(sp); sk = 1 - met["RMSE"] / metrics_from(es[:, hi], ms[:, hi], rating)["RMSE"]
        elif arm == "A6_oracle":
            spk = next((v for kk, v in SMZ.items() if kk[0] == "SP" and kk[2] == k and kk[3] == o), None)
            sk = 1 - met["RMSE"] / metrics_from(*[x[:, hi] for x in kw_err(spk)], rating)["RMSE"] if spk is not None else np.nan
        else:
            sk = np.nan
        rows.append(dict(arm=arm, backbone=bb, target=k, origin=o, L=Ls, horizon_min=15 * h, n_seeds=z.get("n_seeds", 1), skill_vs_SP=sk, **met))
for (arm, bb, k, o, Ls, s), z in RES.items():
    for hi, h in enumerate(HORIZONS):
        e, m = kw_err(z); seed_rows.append(dict(arm=arm, backbone=bb, target=k, origin=o, L=Ls, seed=s, horizon_min=15 * h, **metrics_from(e[:, hi], m[:, hi], META[k]["rating_kw"])))
MAIN = pd.DataFrame(rows); SEEDM = pd.DataFrame(seed_rows)
if len(MAIN):
    save_csv(MAIN.sort_values(["L", "backbone", "arm", "target", "origin", "horizon_min"]), DIRS["tables"] / "main_results.csv")
    save_csv(SEEDM, DIRS["tables"] / "per_seed_metrics.csv")
    AGG = (MAIN[MAIN.arm.isin(ARMS + ["SP", "A6_oracle"])].groupby(["arm", "backbone", "L", "horizon_min"])
           .agg(RMSE=("RMSE", "mean"), nRMSE=("nRMSE", "mean"), MAE=("MAE", "mean"), skill_vs_SP=("skill_vs_SP", "mean"), cells=("RMSE", "size")).reset_index())
    save_csv(AGG, DIRS["tables"] / "main_results_aggregate.csv")

# ---- daily loss differentials, Diebold–Mariano (Newey–West), stratified block bootstrap ----
def daily_sq(z, hi):
    e, m = kw_err(z); t = pd.to_datetime(z["t"]).normalize()
    df = pd.DataFrame({"d": t[m[:, hi]], "se": (e[:, hi] ** 2)[m[:, hi]]})
    return df.groupby("d")["se"].agg(["sum", "count"])

def dm_test(diff):
    d = np.asarray(diff, float); d = d[np.isfinite(d)]; n = len(d)
    if n < 5 or np.allclose(d, d[0]):
        return np.nan, np.nan
    lag = int(np.floor(n ** (1 / 3))); dc = d - d.mean(); g0 = np.mean(dc * dc)
    lrv = g0 + 2 * sum((1 - j / (lag + 1)) * np.mean(dc[j:] * dc[:-j]) for j in range(1, lag + 1))
    if lrv <= 0:
        return np.nan, np.nan
    stat = d.mean() / math.sqrt(lrv / n)
    return float(stat), float(2 * sst.t.sf(abs(stat), n - 1))

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = np.isfinite(p); idx = np.where(ok)[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]; m_ = len(order); run = 0.0
    for i, j in enumerate(order):
        run = max(run, min(1.0, (m_ - i) * p[j])); out[j] = run
    return out

# ---- metric self-test against hand-calculated values (a mismatch is a software FAIL) ----
def metric_self_test():
    rows = []
    def chk(name, got, want, tol=1e-6):
        rows.append(dict(check=name, got=float(got), expected=float(want), ok=bool(np.isfinite(got) and abs(got - want) <= tol)))
    # kW errors: pred [0.5, 0.8, 0.2], truth [0.4, 1.0, 0.2], clear-sky kW [2, 3, 1], mask [T, T, F] → errors [+0.2, −0.6]
    z = dict(pred=np.array([[0.5], [0.8], [0.2]]), y=np.array([[0.4], [1.0], [0.2]]), pcs=np.array([[2.0], [3.0], [1.0]]), m=np.array([[True], [True], [False]]))
    e, m = kw_err(z); met = metrics_from(e[:, 0], m[:, 0], rating=2.0)
    chk("RMSE = sqrt((0.04+0.36)/2) = 0.4472136", met["RMSE"], 0.4472136, 1e-6)
    chk("MAE = (0.2+0.6)/2 = 0.4", met["MAE"], 0.4); chk("MBE = (0.2−0.6)/2 = −0.2", met["MBE"], -0.2)
    chk("nRMSE = 0.4472136/2 = 0.2236068", met["nRMSE"], 0.2236068, 1e-6)
    chk("skill vs reference RMSE 0.5 = 1 − 0.4472136/0.5 = 0.1055728", 1 - met["RMSE"] / 0.5, 0.1055728, 1e-6)
    # Diebold–Mariano, d = 1..6: mean 3.5, γ0 = 17.5/6, γ1 = 8.75/5, lag 1 → LRV = 4.6666667, stat = 3.5/sqrt(4.6666667/6) = 3.9686270
    st, _ = dm_test(np.arange(1, 7, dtype=float)); chk("DM statistic (d = 1..6) = 3.9686270", st, 3.9686270, 1e-5)
    # Holm, p = [0.01, 0.04, 0.03] → [0.03, 0.06, 0.06]
    hp = holm([0.01, 0.04, 0.03]); [chk(f"Holm p[{i}]", hp[i], w) for i, w in enumerate([0.03, 0.06, 0.06])]
    df = pd.DataFrame(rows); save_csv(df, DIRS["tables"] / "metric_self_test.csv", quiet=True)
    return bool(df.ok.all()), df
METRIC_SELFTEST_OK, METRIC_SELFTEST = metric_self_test()
print("metric self-test:", "PASS" if METRIC_SELFTEST_OK else "FAIL\n" + METRIC_SELFTEST[~METRIC_SELFTEST.ok].to_string(index=False))

def paired_days(a, b, bb, Ls, hi):
    """Stack daily sums for arm a and arm b over all targets/origins (same rows), keyed by group for stratified bootstrap."""
    parts = []
    for (arm, bb_, k, o, L_), z in SMZ.items():
        if arm == a and bb_ == bb and L_ == Ls:
            zb = SMZ.get((b, bb if b != "SP" else "ref", k, o, Ls))
            if zb is None:
                continue
            da, db = daily_sq(z, hi), daily_sq(zb, hi)
            j = da.join(db, lsuffix="_a", rsuffix="_b", how="inner"); j["grp"] = f"{k}|{o}"
            parts.append(j)
    return pd.concat(parts) if parts else None

RNG = np.random.RandomState(12345)
def boot_ratio(J, B, block=7):
    """RMSE ratio a/b with a circular 7-day block bootstrap stratified by target×origin; returns point, 90% CI, 95% CI."""
    point = math.sqrt(J.sum_a.sum() / J.sum_b.sum())
    groups = [g.reset_index(drop=True) for _, g in J.groupby("grp")]
    draws = []
    for _ in range(B):
        sa = sb = 0.0
        for g in groups:
            n = len(g); nb = max(1, math.ceil(n / block)); starts = RNG.randint(0, n, nb)   # circular block bootstrap
            ix = np.concatenate([(s + np.arange(block)) % n for s in starts])[:n]
            sa += g.sum_a.values[ix].sum(); sb += g.sum_b.values[ix].sum()
        draws.append(math.sqrt(sa / sb) if sb > 0 else np.nan)
    d = np.array(draws)
    return point, np.nanpercentile(d, [5, 95]), np.nanpercentile(d, [2.5, 97.5]), d

COMPS = [("H1", "A3_ddpm", "A1_replicate")] + [("H2", "A5_transfer", "A0_target")] + \
        [("vsA0", a, "A0_target") for a in ARMS if a != "A0_target"] + [("vsSP", a, "SP") for a in ARMS]
TESTS, BOOT = [], {}
for fam, a, b in COMPS:
    for bb in BACKBONES:
        for L in CFG["lengths"]:
            if fam == "H2" and L not in (14, 60):
                continue
            for hi, h in enumerate(HORIZONS):
                J = paired_days(a, b, bb, f"L{L}", hi)
                if J is None or len(J) < 5:
                    TESTS.append(dict(family=fam, arm_a=a, arm_b=b, backbone=bb, L=L, horizon_min=15 * h, n_days=0, note="insufficient data")); continue
                diff = (J.sum_a / J.count_a - J.sum_b / J.count_b).values
                stat, p = dm_test(diff)
                pt, ci90, ci95, draws = boot_ratio(J, CFG["boot"])
                BOOT[(fam, a, b, bb, L, h)] = (J, draws)
                TESTS.append(dict(family=fam, arm_a=a, arm_b=b, backbone=bb, L=L, horizon_min=15 * h, n_days=len(J),
                                  rmse_ratio=pt, ci90_lo=ci90[0], ci90_hi=ci90[1], ci95_lo=ci95[0], ci95_hi=ci95[1], dm_stat=stat, p=p,
                                  equivalent_2pct=bool(1 - EQUIV_MARGIN <= ci90[0] and ci90[1] <= 1 + EQUIV_MARGIN)))
TESTS = pd.DataFrame(TESTS)
if len(TESTS) and "p" in TESTS:
    TESTS["p_holm"] = np.nan
    for fam in TESTS.family.unique():
        ix = TESTS.family == fam; TESTS.loc[ix, "p_holm"] = holm(TESTS.loc[ix, "p"].values)
    save_csv(TESTS, DIRS["tables"] / "statistical_tests.csv")

# ---- D2: remedy × backbone interaction (paired days, same bootstrap draws) ----
INTER = []
for a in ARMS[1:]:
    for L in CFG["lengths"]:
        for hi, h in enumerate(HORIZONS):
            kl, kg = ("vsA0", a, "A0_target", "lstm", L, h), ("vsA0", a, "A0_target", "lgbm", L, h)
            if kl in BOOT and kg in BOOT:
                gl, gg = 1 - BOOT[kl][1], 1 - BOOT[kg][1]
                pl = 1 - math.sqrt(BOOT[kl][0].sum_a.sum() / BOOT[kl][0].sum_b.sum()); pg = 1 - math.sqrt(BOOT[kg][0].sum_a.sum() / BOOT[kg][0].sum_b.sum())
                diff = gl - gg
                INTER.append(dict(arm=a, L=L, horizon_min=15 * h, gain_lstm=pl, gain_lgbm=pg, interaction=pl - pg,
                                  ci95_lo=np.nanpercentile(diff, 2.5), ci95_hi=np.nanpercentile(diff, 97.5),
                                  note="independent bootstrap draws per backbone (conservative)"))
INTER = pd.DataFrame(INTER)
if len(INTER):
    save_csv(INTER, DIRS["tables"] / "interaction_backbone.csv")

# ---- D7: crossover data length (gain vs A0 with 95% CI) ----
CROSS = []
if len(TESTS) and "ci95_lo" in TESTS:
    for (a, bb, h), g in TESTS[TESTS.family == "vsA0"].groupby(["arm_a", "backbone", "horizon_min"]):
        g = g.sort_values("L"); helps = list(g.ci95_hi < 1.0)
        Ls_ = list(g.L)
        if all(helps):
            verdict = f"helps at every length tested ({Ls_[0]}–{Ls_[-1]} d)"
        elif not any(helps):
            verdict = "does not help at any length tested"
        else:
            first_stop = next((L_ for L_, hp in zip(Ls_, helps) if not hp and any(helps[: Ls_.index(L_)])), None)
            verdict = f"stops helping by {first_stop} d" if first_stop else "mixed"
        CROSS.append(dict(arm=a, backbone=bb, horizon_min=h, **{f"gain_L{L_}": 1 - r for L_, r in zip(g.L, g.rmse_ratio)},
                          **{f"helps_L{L_}": hp for L_, hp in zip(g.L, helps)}, verdict=verdict))
CROSS = pd.DataFrame(CROSS)
if len(CROSS):
    save_csv(CROSS, DIRS["tables"] / "crossover_length.csv")

# ---- H3: rank by length (h = 15 min primary) ----
RANK = []
if len(MAIN):
    for (k, bb, Ls), g in MAIN[(MAIN.arm.isin(ARMS)) & (MAIN.horizon_min == 15 * HORIZONS[0])].groupby(["target", "backbone", "L"]):
        gg = g.groupby("arm").RMSE.mean().sort_values()
        for r_, (arm, v) in enumerate(gg.items(), 1):
            RANK.append(dict(target=k, backbone=bb, L=Ls, arm=arm, rank=r_, RMSE=v))
RANK = pd.DataFrame(RANK)
if len(RANK):
    save_csv(RANK, DIRS["tables"] / "rank_by_length.csv")

# ---- subgroups: season (origin), variability tercile (cut points from the pre-origin year), horizon ----
SUB = []
for (arm, bb, k, o, Ls), z in SMZ.items():
    if arm not in ARMS + ["SP"]:
        continue
    hist = ARR[k].loc[pd.Timestamp(o) - pd.Timedelta(days=365): pd.Timestamp(o) - pd.Timedelta("1min"), "k"]
    dsd = hist.groupby(hist.index.normalize()).std().dropna()
    if len(dsd) < 10:
        continue
    q1, q2 = np.percentile(dsd, [100 / 3, 200 / 3])
    obs = pd.Series(z["y"][:, 0], index=pd.to_datetime(z["t"])).where(z["m"][:, 0])
    tsd = obs.groupby(obs.index.normalize()).std()
    terc = tsd.map(lambda v: "low" if v <= q1 else ("mid" if v <= q2 else "high"))
    season = "summer (Jan origin)" if pd.Timestamp(o).month in (12, 1, 2) else "winter (Jul origin)" if pd.Timestamp(o).month in (6, 7, 8) else f"month {pd.Timestamp(o).month}"
    for hi, h in enumerate(HORIZONS):
        e, m = kw_err(z); df = pd.DataFrame({"d": pd.to_datetime(z["t"]).normalize(), "se": e[:, hi] ** 2, "m": m[:, hi]})
        df = df[df.m]; df["terc"] = df.d.map(terc)
        for tg, gg in df.groupby("terc"):
            SUB.append(dict(arm=arm, backbone=bb, target=k, origin=o, season=season, L=Ls, horizon_min=15 * h, variability=tg,
                            RMSE=float(np.sqrt(gg.se.mean())), n=len(gg)))
SUB = pd.DataFrame(SUB)
if len(SUB):
    save_csv(SUB, DIRS["tables"] / "subgroup_results.csv")

# ---- Phase 0 feasibility (pilot only): headroom = (A0 at shortest length − oracle) / A0 ----
FEAS = []
if CFG["oracle"] and len(MAIN):
    Lmin = f"L{min(CFG['lengths'])}"
    for bb in BACKBONES:
        for hi, h in enumerate(HORIZONS):
            a0 = MAIN[(MAIN.arm == "A0_target") & (MAIN.backbone == bb) & (MAIN.L == Lmin) & (MAIN.horizon_min == 15 * h)]
            orc = MAIN[(MAIN.arm == "A6_oracle") & (MAIN.backbone == bb) & (MAIN.horizon_min == 15 * h)]
            if len(a0) and len(orc):
                agg = (a0.RMSE.mean() - orc.RMSE.mean()) / a0.RMSE.mean()
                tails = []
                for (arm, bb_, k, o, Ls), z in SMZ.items():
                    if bb_ == bb and ((arm == "A0_target" and Ls == Lmin) or arm == "A6_oracle"):
                        obs = pd.Series(z["y"][:, hi], index=pd.to_datetime(z["t"])).where(z["m"][:, hi])
                        sdd = obs.groupby(obs.index.normalize()).std(); top = sdd[sdd >= sdd.quantile(0.9)].index
                        e, m = kw_err(z); d = pd.to_datetime(z["t"]).normalize(); sel = m[:, hi] & np.isin(d, top)
                        tails.append(dict(arm=arm, rmse=float(np.sqrt(np.mean(e[sel, hi] ** 2))) if sel.any() else np.nan))
                tl = pd.DataFrame(tails); ta = tl[tl.arm == "A0_target"].rmse.mean(); to = tl[tl.arm == "A6_oracle"].rmse.mean()
                FEAS.append(dict(backbone=bb, horizon_min=15 * h, rmse_A0_shortest=a0.RMSE.mean(), rmse_oracle=orc.RMSE.mean(),
                                 headroom_aggregate=agg, headroom_tail_top10pct_days=(ta - to) / ta if ta else np.nan,
                                 A0_skill_at_longest=MAIN[(MAIN.arm == "A0_target") & (MAIN.backbone == bb) & (MAIN.L == f"L{max(CFG['lengths'])}") & (MAIN.horizon_min == 15 * h)].skill_vs_SP.mean(),
                                 window="pilot test window (2022), exploratory"))
FEAS = pd.DataFrame(FEAS)
if len(FEAS):
    save_csv(FEAS, DIRS["tables"] / "feasibility_headroom.csv")

# ---- compute cost per arm (from task files) ----
cost = []
for name, r in LEDGER.rows.items():
    if r["kind"] == "model" and r["status"] == "COMPLETED":
        arm, bb, k, o, Ls, ss = name.split("__"); z = load_task(name)
        if z is not None:
            cost.append(dict(arm=arm, backbone=bb, L=Ls, seconds=z["meta"].get("seconds", np.nan), n_train=z["meta"].get("n_train", np.nan),
                             n_params=z["meta"].get("n_params", -1)))
COSTDF = pd.DataFrame(cost)
if len(COSTDF):
    gen_s = pd.DataFrame(GEN_VALID).seconds.mean() if GEN_VALID else np.nan
    CT = COSTDF.groupby(["arm", "backbone", "L"]).agg(train_seconds_mean=("seconds", "mean"), n_train_mean=("n_train", "mean")).reset_index()
    CT["generator_seconds_mean"] = np.where(CT.arm == "A3_ddpm", gen_s, 0.0)
    save_csv(CT, DIRS["tables"] / "compute_cost.csv")
# ---- sensitivity table (exploratory): augmentation amount 0.25 / 0.50 (main) / 1.00 ----
SENS = []
for name, r in LEDGER.rows.items():
    if r["kind"] != "sensitivity" or r["status"] != "COMPLETED":
        continue
    tag, bb, k_, o, Ls, ss = name.split("__"); _, arm, f = tag.split("-", 2)
    k = next((t for t in TARGETS if tid_name(t) == k_), k_); z = load_task(name)
    main = RES.get((arm, bb, k, o, Ls, int(ss[1:]))); sp = SMZ.get(("SP", "ref", k, o, Ls))
    if z is None or main is None:
        continue
    for hi, h in enumerate(HORIZONS):
        rating = META[k]["rating_kw"]; spr = metrics_from(*[x[:, hi] for x in kw_err(sp)], rating)["RMSE"] if sp is not None else np.nan
        rs = metrics_from(*[x[:, hi] for x in kw_err(z)], rating)["RMSE"]; rm = metrics_from(*[x[:, hi] for x in kw_err(main)], rating)["RMSE"]
        SENS.append(dict(arm=arm, backbone=bb, target=k, origin=o, L=Ls, seed=int(ss[1:]), horizon_min=15 * h, aug_frac=float(f[1:]),
                         RMSE=rs, skill_vs_SP=1 - rs / spr if spr else np.nan, RMSE_main_frac_0p50=rm, change_vs_main=rs / rm - 1, label="exploratory"))
SENS = pd.DataFrame(SENS)
if len(SENS):
    save_csv(SENS.sort_values(["arm", "backbone", "target", "horizon_min", "aug_frac"]), DIRS["tables"] / "sensitivity_aug_frac.csv")

# ---- forecasts for independent recomputation (every completed task; float32; truth once per split) ----
FC = {}; FC_INDEX = []
for name, r in LEDGER.rows.items():
    if r["status"] != "COMPLETED" or r["kind"] not in ("model", "reference", "oracle", "sensitivity"):
        continue
    z = load_task(name)
    if z is None:
        continue
    arm, bb, k_, o, Ls, ss = name.split("__"); split = f"{k_}__{o}__{Ls}"
    if f"truth__{split}__y" not in FC:
        FC[f"truth__{split}__t_ns"] = z["t"]; FC[f"truth__{split}__y"] = z["y"].astype("float32")
        FC[f"truth__{split}__clearsky_kw"] = z["pcs"].astype("float32"); FC[f"truth__{split}__mask"] = z["m"]
    FC[f"pred__{name}"] = z["pred"].astype("float32")
    FC_INDEX.append(dict(task=name, kind=r["kind"], arm=arm, backbone=bb, target=k_, origin=o, L=Ls, seed=ss, split=split))
if FC:
    np.savez_compressed(DIRS["predictions"] / f"forecasts_{MODE}.npz", **FC)
    save_csv(pd.DataFrame(FC_INDEX), DIRS["predictions"] / "forecast_index.csv", quiet=True)
    atomic_write_text(DIRS["predictions"] / "README_forecasts.md",
        f"# Forecasts ({MODE}, run {RUN_ID})\n\nFile `forecasts_{MODE}.npz` (numpy). Arrays per split `<target>__<origin>__<L>`: `truth__<split>__t_ns` (issue time, "
        "local standard time, ns), `truth__<split>__y` (observed clear-sky index, one column per horizon: "
        f"{', '.join(str(15*h) + ' min' for h in HORIZONS)}), `truth__<split>__clearsky_kw` (clear-sky power, kW), `truth__<split>__mask` (daytime rows "
        "that count). Per task `pred__<task>` (forecast clear-sky index, same shape). `forecast_index.csv` lists every task.\n\n"
        "Recompute: kW error = (pred − y) × clearsky_kw on mask; RMSE = sqrt(mean(error²)). Seed-mean forecasts are the mean of `pred` over seeds. "
        "Ratings for nRMSE are in `data_audit/DATA_PROVENANCE.json` (meta).\n")
    # recomputation check: every MAIN row for one task family is reproduced from the saved file alone
    _fz = np.load(DIRS["predictions"] / f"forecasts_{MODE}.npz"); _bad = 0; _n = 0
    for r in SEEDM.head(50).itertuples():
        nm = tid_name(r.arm, r.backbone, r.target, r.origin, r.L, f"s{r.seed}"); split = f"{tid_name(r.target)}__{r.origin}__{r.L}"
        if f"pred__{nm}" not in _fz:
            continue
        hi = HORIZONS.index(r.horizon_min // 15); mk = _fz[f"truth__{split}__mask"][:, hi]
        e = ((_fz[f"pred__{nm}"][:, hi] - _fz[f"truth__{split}__y"][:, hi]) * _fz[f"truth__{split}__clearsky_kw"][:, hi])[mk]
        _n += 1; _bad += int(abs(math.sqrt(np.mean(e.astype("float64") ** 2)) - r.RMSE) > 1e-4 * max(1.0, r.RMSE))
    FORECAST_RECOMPUTE_OK = _n > 0 and _bad == 0
    print(f"forecast recomputation check: {_n - _bad}/{_n} metrics reproduced from the saved forecasts")
else:
    FORECAST_RECOMPUTE_OK = False

# ---- reproduction gate (only when REPRO_REF_ZIP is set): clean-session rerun vs an earlier results ZIP ----
REPRO_ROWS = []
if REPRO:
    try:
        with zipfile.ZipFile(REPRO_REF_ZIP) as _zf:
            _nm = next(n for n in _zf.namelist() if n.startswith("predictions/forecasts_") and n.endswith(".npz"))
            _zf.extract(_nm, OUT / "repro_ref"); _ref = np.load(OUT / "repro_ref" / _nm)
        for name in [n for n in FC if n.startswith("pred__")]:
            if name not in _ref:
                REPRO_ROWS.append(dict(task=name[6:], status="not in reference ZIP")); continue
            split = "__".join(name[6:].split("__")[2:5]); mk = FC[f"truth__{split}__mask"]
            for hi, h in enumerate(HORIZONS):
                e_new = ((FC[name][:, hi] - FC[f"truth__{split}__y"][:, hi]) * FC[f"truth__{split}__clearsky_kw"][:, hi])[mk[:, hi]]
                e_ref = ((_ref[name][:, hi] - _ref[f"truth__{split}__y"][:, hi]) * _ref[f"truth__{split}__clearsky_kw"][:, hi])[_ref[f"truth__{split}__mask"][:, hi]]
                r_new, r_ref = math.sqrt(np.mean(e_new ** 2)), math.sqrt(np.mean(e_ref ** 2))
                REPRO_ROWS.append(dict(task=name[6:], horizon_min=15 * h, rmse_rerun=r_new, rmse_reference=r_ref, rel_diff=r_new / r_ref - 1,
                                       within_tol=bool(abs(r_new / r_ref - 1) <= REPRO_TOL), status="compared"))
    except Exception as e:
        REPRO_ROWS.append(dict(task="-", status=f"reference ZIP unusable: {type(e).__name__}: {e}"))
    save_csv(pd.DataFrame(REPRO_ROWS), DIRS["tables"] / "reproduction_check.csv")
log_runtime("evaluation", time.time() - t_eval)


In [ ]:
# ===== CELL 9 — BASE-PAPER REPLICATION (H4): base protocol vs equal-update protocol =====
# Base protocol (Demir & Najaf 2026): window 5, LSTM 32 → Dense 8 ReLU → Dropout 0.2 → 1, Adam 1e-4, batch 16,
# early stopping (patience 10), LR on plateau (0.5, patience 5); DDPM samples (+50%) added to train AND validation.
# Equal protocol (this paper): synthetic rows enter training only; every arm gets the same number of optimizer updates.
t_rep = time.time()

class RepNet(nn.Module):
    def __init__(self):
        super().__init__(); self.lstm = nn.LSTM(1, 32, batch_first=True)
        self.head = nn.Sequential(nn.Linear(32, 8), nn.ReLU(), nn.Dropout(0.2), nn.Linear(8, 1))
    def forward(self, x):
        _, (h, _) = self.lstm(x.unsqueeze(-1)); return self.head(h[-1]).squeeze(-1)

def windows(v, w=5):
    v = np.asarray(v, "float32"); idx = np.arange(len(v) - w)
    X = np.stack([v[i:i + w] for i in idx]) if len(idx) else np.zeros((0, w), "float32")
    y = v[w:] if len(v) > w else np.zeros(0, "float32")
    ok = np.isfinite(X).all(1) & np.isfinite(y)
    return X[ok], y[ok]

def rep_train(Xtr, ytr, Xva, yva, seed, epochs=None, updates=None):
    set_seed(seed); net = RepNet().to(DEVICE); opt = torch.optim.Adam(net.parameters(), lr=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=5, min_lr=1e-6)
    Xt, yt, Xv, yv = _t(Xtr), _t(ytr), _t(Xva), _t(yva); n = len(yt); rng = np.random.RandomState(seed)
    best, best_state, wait, used = float("inf"), None, 0, 0
    if updates is None:                                  # base protocol: epochs over the (possibly enlarged) set
        for ep in range(epochs):
            perm = rng.permutation(n); net.train()
            for i in range(0, n, 16):
                b = torch.as_tensor(perm[i:i + 16], device=DEVICE); loss = ((net(Xt[b]) - yt[b]) ** 2).mean()
                opt.zero_grad(); loss.backward(); opt.step(); used += 1
            net.eval()
            with torch.no_grad():
                vl = ((net(Xv) - yv) ** 2).mean().item()
            sched.step(vl)
            if vl < best:
                best, best_state, wait = vl, {k: v.clone() for k, v in net.state_dict().items()}, 0
            else:
                wait += 1
                if wait >= 10:
                    break
    else:                                                # equal protocol: exactly `updates` steps, real-only validation
        every = max(1, updates // 20)
        for u in range(1, updates + 1):
            net.train(); b = torch.as_tensor(rng.randint(0, n, 16), device=DEVICE)
            loss = ((net(Xt[b]) - yt[b]) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step(); used += 1
            if u % every == 0 or u == updates:
                net.eval()
                with torch.no_grad():
                    vl = ((net(Xv) - yv) ** 2).mean().item()
                if vl < best:
                    best, best_state = vl, {k: v.clone() for k, v in net.state_dict().items()}
    net.load_state_dict(best_state); net.eval()
    return net, used

def rep_ddpm(X, y, n_new, seed):
    D = dict(seq=X, stat=np.zeros((len(X), 0), "float32"), y=y[:, None], m=np.ones((len(X), 1), bool), rating=1.0)
    old = (LAGS, HORIZONS)
    globals()["LAGS"], globals()["HORIZONS"] = X.shape[1], [1]
    try:
        E, _ = train_sample_ddpm(D, n_new, CFG["ddpm_iters"], seed)
    finally:
        globals()["LAGS"], globals()["HORIZONS"] = old
    return E["seq"], E["y"][:, 0]

def rep_case(name, tr, va, te, seeds):
    """One replication case. Each seed is saved as soon as it finishes and reused on rerun (LONG-STAGE RULE)."""
    Xtr, ytr = windows(tr); Xva, yva = windows(va); Xte, yte = windows(te)
    if min(len(ytr), len(yva), len(yte)) < 20:
        raise ValueError(f"{name}: too few windows (train {len(ytr)}, val {len(yva)}, test {len(yte)})")
    out, t_case = [], time.time()
    print(f"  [replication] {name}: {len(ytr):,} training windows, {len(seeds)} seed(s), up to {CFG['rep_epochs']} epochs at batch 16")
    for j, s in enumerate(seeds):
        fp = DIRS["tasks"] / f"REP_{name}_s{s}.json"
        if fp.exists():
            try:
                r = json.loads(fp.read_text()); assert all(np.isfinite(r[k]) for k in ["base_B0", "base_B3", "eq_E0", "eq_E3", "eq_E2"])
                out.append(r); print(f"  [replication] {name} seed {s}: reused saved result"); continue
            except Exception:
                fp.unlink(missing_ok=True)
        r = {}; t_s = time.time()
        net, u0 = rep_train(Xtr, ytr, Xva, yva, s, epochs=CFG["rep_epochs"]); r["base_B0"] = net; r["u_B0"] = u0
        print(f"  [replication] {name} seed {s}: base B0 done ({u0:,} updates, {(time.time() - t_s) / 60:.1f} min)")
        Xs, ys = rep_ddpm(np.r_[Xtr, Xva], np.r_[ytr, yva], int(AUG_FRAC * (len(ytr) + len(yva))), s)
        ntr_s = int(AUG_FRAC * len(ytr))
        net, u3 = rep_train(np.r_[Xtr, Xs[:ntr_s]], np.r_[ytr, ys[:ntr_s]], np.r_[Xva, Xs[ntr_s:]], np.r_[yva, ys[ntr_s:]], s, epochs=CFG["rep_epochs"])
        r["base_B3"] = net; r["u_B3"] = u3
        print(f"  [replication] {name} seed {s}: base B3 (DDPM) done ({u3:,} updates, {(time.time() - t_s) / 60:.1f} min)")
        U = CFG["rep_epochs"] * math.ceil(len(ytr) / 16)
        Xs2, ys2 = rep_ddpm(Xtr, ytr, ntr_s, s + 1000)
        r["eq_E0"], _ = rep_train(Xtr, ytr, Xva, yva, s, updates=U)
        r["eq_E3"], _ = rep_train(np.r_[Xtr, Xs2], np.r_[ytr, ys2], Xva, yva, s, updates=U)
        rng = np.random.RandomState(s); ix = rng.choice(len(ytr), ntr_s)
        r["eq_E2"], _ = rep_train(np.r_[Xtr, np.clip(Xtr[ix] + rng.normal(0, JITTER_SIGMA, Xtr[ix].shape) * Xtr.std(0), 0, None)],
                                  np.r_[ytr, ytr[ix]], Xva, yva, s, updates=U)
        with torch.no_grad():
            for key in ["base_B0", "base_B3", "eq_E0", "eq_E3", "eq_E2"]:
                p = r[key](_t(Xte)).cpu().numpy(); r[key] = float(np.sqrt(np.mean((p - yte) ** 2)))
        r = dict(case=name, seed=s, **r); atomic_write_text(fp, json.dumps(r)); out.append(r)
        done = j + 1; el = time.time() - t_case
        print(f"  [replication] {name}: seed {s} saved | {done}/{len(seeds)} seeds ({100 * done / len(seeds):.0f}%) | "
              f"elapsed {el / 60:.1f} min | ETA {el / done * (len(seeds) - done) / 60:.1f} min")
        checkpoint_zip()                                  # saved seed goes into the checkpoint (and to Drive when on)
    return pd.DataFrame(out)

REP = []; REP_STATUS = {}
kg = INV[INV.kind == "kaggle_gen"]
LEDGER.expect("REP_kaggle", "replication", required=len(kg) >= 2 and not REPRO and RUN_REPLICATION)   # optional files: required only when uploaded
LEDGER.expect("REP_yulara", "replication", required=REPLICATION_ONLY and len(INV[INV.kind == "yulara_array"]) > 0)
try:
    if REPRO:
        raise LookupError("reproduction check: replication not rerun")
    if not RUN_REPLICATION:
        raise LookupError(f"replication switched off in {MODE} (RUN_REPLICATION = False); H4 is reported from STANDARD_RUN")
    if len(kg) < 2:
        raise LookupError("optional Kaggle Plant_1/Plant_2 files not in general.zip")
    def plant_series(f):
        d = pd.read_csv(DATA_DIR / f); d["ts"] = parse_time(d["DATE_TIME"])
        d["DAILY_YIELD"] = clean_sentinels(d["DAILY_YIELD"].values)[0]
        return (d.groupby("ts")["DAILY_YIELD"].sum() / 1000.0).sort_index().resample("15min").mean()
    f1 = kg[kg.file.str.lower().str.contains("plant_1")].file.iloc[0]; f2 = kg[kg.file.str.lower().str.contains("plant_2")].file.iloc[0]
    s1, s2 = plant_series(f1), plant_series(f2)
    lo, hi = float(s1.min()), float(s1.max())                     # base protocol: MinMax fitted on Plant 1, applied to Plant 2
    n1 = (s1 - lo) / (hi - lo); n2 = (s2 - lo) / (hi - lo)
    cut = min(2900, int(len(n1) * 0.89))
    REP.append(rep_case("kaggle_P1_to_P2", n1.values[:cut], n1.values[cut:], n2.values, CFG["rep_seeds"]))
    LEDGER.done("REP_kaggle", seconds=time.time() - t_rep)
except LookupError as e:
    LEDGER.na("REP_kaggle", str(e))
except Exception as e:
    LEDGER.fail("REP_kaggle", f"{type(e).__name__}: {e}")
try:
    ya = INV[INV.kind == "yulara_array"]
    if REPRO or not RUN_REPLICATION:
        LEDGER.na("REP_yulara", "replication not run in this mode")
    elif len(ya) == 0:
        LEDGER.na("REP_yulara", "optional Yulara files not uploaded")
    else:
        ys_, _, _ = load_series(DATA_DIR / ya.file.iloc[0], MAXDATE, {"P": (("activepower",), ())}, start="2016-01-01")
        p = ys_["P"].clip(lower=0)
        if MODE == "FULL_PAPER_RUN":
            tr, va, te = p[:"2021-12-31"], p["2022-01-01":"2022-12-31"], p["2023-01-01":"2023-12-31"]
        else:                                                         # stand-in split: no 2023 data before FULL
            tr, va, te = p[:"2020-12-31"], p["2021-01-01":"2021-12-31"], p["2022-01-01":"2022-12-31"]
        lo, hi = float(tr.min()), float(tr.max())
        REP.append(rep_case("yulara_seasonal", ((tr - lo) / (hi - lo)).values, ((va - lo) / (hi - lo)).values, ((te - lo) / (hi - lo)).values, CFG["rep_seeds"][:YULARA_REP_SEEDS]))
        LEDGER.done("REP_yulara")
except Exception as e:
    LEDGER.fail("REP_yulara", f"{type(e).__name__}: {e}")
REPDF = pd.concat(REP) if REP else pd.DataFrame()
if len(REPDF):
    s = REPDF.groupby("case").agg(B0_base_rmse=("base_B0", "mean"), B0_sd=("base_B0", "std"), B3_ddpm_base_rmse=("base_B3", "mean"), B3_sd=("base_B3", "std"),
                                  E0_equal_rmse=("eq_E0", "mean"), E3_ddpm_equal_rmse=("eq_E3", "mean"), E2_jitter_equal_rmse=("eq_E2", "mean"),
                                  updates_B0=("u_B0", "mean"), updates_B3=("u_B3", "mean"), seeds=("seed", "nunique")).reset_index()
    s["gain_base_protocol"] = 1 - s.B3_ddpm_base_rmse / s.B0_base_rmse
    s["gain_equal_protocol"] = 1 - s.E3_ddpm_equal_rmse / s.E0_equal_rmse
    s["gain_jitter_equal"] = 1 - s.E2_jitter_equal_rmse / s.E0_equal_rmse
    s["units"] = "scaled target (MinMax), all intervals, as in the base paper"
    save_csv(REPDF, DIRS["tables"] / "replication_per_seed.csv"); save_csv(s, DIRS["tables"] / "replication_base_paper.csv")
LEDGER.save(); log_runtime("replication", time.time() - t_rep)


In [ ]:
# ===== CELL 10 — FIGURES (drawn only from saved tables; 300-dpi PNG + PDF) =====
FIGS = []
def savefig(fig, stem):
    for ext in ("png", "pdf"):
        p = DIRS["figures"] / f"{stem}.{ext}"; fig.savefig(p, dpi=300, bbox_inches="tight")
    plt.close(fig); FIGS.append(stem); print(f"  saved figures/{stem}.png/.pdf")
try:
    agg_p = DIRS["tables"] / "main_results_aggregate.csv"
    if agg_p.exists():
        A = pd.read_csv(agg_p); A["Ln"] = A.L.astype(str).str.extract(r"(\d+)").astype(float)
        for bb in BACKBONES:
            g = A[(A.backbone == bb) & A.arm.isin(ARMS) & (A.horizon_min == 15 * HORIZONS[0])]
            if len(g):
                fig, ax = plt.subplots(figsize=(6, 4))
                for arm, gg in g.groupby("arm"):
                    gg = gg.sort_values("Ln"); ax.plot(gg.Ln, gg.skill_vs_SP, marker="o", label=arm)
                ax.axhline(0, color="grey", lw=0.8); ax.set_xscale("log"); ax.set_xlabel("days of local data (log scale)")
                ax.set_ylabel("skill vs smart persistence (15 min)"); ax.set_title(f"{bb.upper()}: skill by data length"); ax.legend(fontsize=7)
                savefig(fig, f"fig_skill_vs_length_{bb}")
    rk = DIRS["tables"] / "rank_by_length.csv"
    if rk.exists():
        R = pd.read_csv(rk); P = R.pivot_table(index="arm", columns=["backbone", "L"], values="rank", aggfunc="mean")
        if P.size:
            fig, ax = plt.subplots(figsize=(1.2 + 0.9 * P.shape[1], 0.5 + 0.45 * P.shape[0]))
            im = ax.imshow(P.values, cmap="viridis_r", aspect="auto"); ax.set_yticks(range(len(P.index)), P.index, fontsize=7)
            ax.set_xticks(range(P.shape[1]), [f"{a}\n{b}" for a, b in P.columns], fontsize=7); fig.colorbar(im, label="mean rank (1 = best)")
            for i in range(P.shape[0]):
                for j in range(P.shape[1]):
                    ax.text(j, i, f"{P.values[i, j]:.1f}", ha="center", va="center", fontsize=6, color="white")
            ax.set_title("Rank of remedies by data length"); savefig(fig, "fig_rank_heatmap")
    if SMZ:
        key = next((kk for kk in SMZ if kk[0] == "A0_target"), None)
        if key:
            z = SMZ[key]; t = pd.to_datetime(z["t"]); n = min(len(t), 96 * 3)
            fig, ax = plt.subplots(figsize=(8, 3)); m = z["m"][:n, 0]
            ax.plot(t[:n][m], (z["y"][:n, 0] * z["pcs"][:n, 0])[m], ".", ms=2, label="observed")
            for arm in ["SP", "A0_target", "A3_ddpm", "A5_transfer"]:
                kk = (arm, "ref" if arm == "SP" else key[1]) + key[2:]
                if kk in SMZ:
                    zz = SMZ[kk]; ax.plot(t[:n][m], (zz["pred"][:n, 0] * zz["pcs"][:n, 0])[m], ".", ms=1.5, label=arm)
            ax.set_ylabel("kW"); ax.set_title(f"15-min forecasts, {key[2]}, first test days ({key[1]}, {key[4]})"); ax.legend(fontsize=6, ncol=5)
            savefig(fig, "fig_forecast_example")
except Exception as e:
    print("⚠ figure error:", type(e).__name__, e)


In [ ]:
# ===== CELL 11 — SOFTWARE CHECK, RESEARCH GATES, VERDICTS, AUDIT, MANIFEST, ZIP =====
if REPLICATION_ONLY:
    REQ_TABLES = ["replication_base_paper.csv", "replication_per_seed.csv"]; REQ_FIGS = []
elif REPRO:
    REQ_TABLES = ["main_results.csv", "per_seed_metrics.csv", "reproduction_check.csv"]; REQ_FIGS = []
else:
    REQ_TABLES = ["main_results.csv", "main_results_aggregate.csv", "per_seed_metrics.csv", "statistical_tests.csv", "interaction_backbone.csv",
                  "crossover_length.csv", "rank_by_length.csv", "subgroup_results.csv", "compute_cost.csv", "synthetic_validity.csv",
                  "sensitivity_aug_frac.csv"] + (["replication_base_paper.csv"] if LEDGER.rows.get("REP_kaggle", {}).get("required") else []) \
                 + (["feasibility_headroom.csv"] if CFG["oracle"] else [])
    REQ_FIGS = [f"fig_skill_vs_length_{bb}" for bb in BACKBONES] + ["fig_rank_heatmap", "fig_forecast_example"]
L = LEDGER.df()
chk = []
def C(name, ok, detail=""):
    chk.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
C("data validation passed", DATA_OK, "; ".join(problems) if not DATA_OK else "")
req = L[L.required] if len(L) else L
bad = req[req.status != "COMPLETED"] if len(req) else req
C("all required tasks completed", len(req) > 0 and len(bad) == 0, f"{len(bad)} of {len(req)} required tasks not completed" +
  (": " + "; ".join(f"{r.task_id} → {r.status} {r.reason}" for r in bad.head(5).itertuples()) if len(bad) else ""))
if REPLICATION_ONLY:
    _rp = DIRS["tables"] / "replication_base_paper.csv"
    C("replication cases completed (Kaggle; Yulara if supplied)", _rp.exists() and len(pd.read_csv(_rp)) >= 1 + int(len(INV[INV.kind == "yulara_array"]) > 0))
elif len(MAIN):
    need = {(k, o, f"L{Lx}", bb, a) for k in TARGETS for o in CFG["origins"] for Lx in CFG["lengths"] for bb in BACKBONES for a in ARMS}
    have = set(zip(MAIN.target, MAIN.origin, MAIN.L, MAIN.backbone, MAIN.arm))
    C("every arm × backbone × split present", need <= have, f"{len(need - have)} missing")
    C("metrics finite", np.isfinite(MAIN[MAIN.arm.isin(ARMS)][["RMSE", "MAE", "skill_vs_SP"]].values).all())
else:
    C("every arm × backbone × split present", False, "no results"); C("metrics finite", False, "no results")
if not REPRO and not REPLICATION_ONLY:
    C("statistics produced", len(TESTS) > 0 and "p" in TESTS and TESTS["p"].notna().any())
C("metric self-test (hand-calculated values)", METRIC_SELFTEST_OK, "" if METRIC_SELFTEST_OK else "; ".join(METRIC_SELFTEST[~METRIC_SELFTEST.ok].check))
if not REPLICATION_ONLY:
    C("forecasts saved and metrics recomputable from them", FORECAST_RECOMPUTE_OK)
C("data provenance: full-file SHA-256 for every file used", bool(len(_used)) and _used.sha256.astype(str).str.len().eq(64).all())
for t in REQ_TABLES:
    p = DIRS["tables"] / t; C(f"table {t}", p.exists() and p.stat().st_size > 0)
for f in REQ_FIGS:
    C(f"figure {f}", (DIRS["figures"] / f"{f}.png").exists())
SOFTWARE = "PASS" if all(c["result"] == "PASS" for c in chk) else "FAIL"
SW = pd.DataFrame(chk); save_csv(SW, DIRS["tables"] / "software_check.csv")

# ---- research gates (pre-declared in the design record) ----
label = "confirmatory" if MODE == "FULL_PAPER_RUN" else "exploratory"
G = []
def gate(h, verdict, evidence):
    G.append(dict(hypothesis=h, verdict=verdict, label=label, evidence=evidence))
T15 = TESTS[TESTS.get("horizon_min", pd.Series(dtype=float)) == 15 * HORIZONS[0]] if len(TESTS) and "horizon_min" in TESTS else pd.DataFrame()
def fam_rows(fam):
    return T15[(T15.family == fam)] if len(T15) else pd.DataFrame()
for fam, hname in [("H1", "H1 DDPM vs replication"), ("H2", "H2 transfer vs target-only")]:
    R = fam_rows(fam)
    for r in R.itertuples():
        if not np.isfinite(getattr(r, "p_holm", np.nan)):
            gate(f"{hname} [{r.backbone}, {r.L} d]", "NOT EVALUABLE", "insufficient days"); continue
        v = "SUPPORTED" if r.p_holm < 0.05 else ("NOT SUPPORTED (equivalent within ±2%)" if r.equivalent_2pct else "NOT SUPPORTED")
        gate(f"{hname} [{r.backbone}, {r.L} d]", v, f"RMSE ratio {r.rmse_ratio:.3f} (95% CI {r.ci95_lo:.3f}–{r.ci95_hi:.3f}), Holm p = {r.p_holm:.3g}")
    if len(R) == 0:
        gate(hname, "NOT EVALUABLE", "no test rows")
if len(RANK) and len(CFG["lengths"]) > 1:
    for bb in BACKBONES:
        rr = RANK[RANK.backbone == bb]; diff = 0; nt = 0
        for k in TARGETS:
            a = rr[(rr.target == k) & (rr.L == f"L{min(CFG['lengths'])}") & (rr["rank"] == 1)].arm.tolist()
            b = rr[(rr.target == k) & (rr.L == f"L{max(CFG['lengths'])}") & (rr["rank"] == 1)].arm.tolist()
            if a and b:
                nt += 1; diff += int(a[0] != b[0])
        need_n = 2 if len(TARGETS) >= 3 else max(1, len(TARGETS))
        gate(f"H3 ranking changes with length [{bb}]", "SUPPORTED" if nt and diff >= need_n else ("NOT SUPPORTED" if nt else "NOT EVALUABLE"),
             f"top arm differs on {diff} of {nt} targets (need ≥ {need_n})")
else:
    gate("H3 ranking changes with length", "NOT EVALUABLE", "only one data length in this mode")
rp = DIRS["tables"] / "replication_base_paper.csv"
if rp.exists():
    for r in pd.read_csv(rp).itertuples():
        ok = r.gain_base_protocol > 0 and r.gain_equal_protocol < r.gain_base_protocol
        gate(f"H4 replication [{r.case}]", "SUPPORTED" if ok else "NOT SUPPORTED",
             f"gain base {r.gain_base_protocol:+.3f} vs equal {r.gain_equal_protocol:+.3f} (jitter equal {r.gain_jitter_equal:+.3f}); seeds {r.seeds}")
else:
    gate("H4 replication", "NOT EVALUABLE", "replication did not run")
if len(INTER):
    ex = INTER[(INTER.ci95_lo > 0) | (INTER.ci95_hi < 0)]
    gate("D2 remedy × backbone interaction", "SUPPORTED" if len(ex) else "NOT SUPPORTED",
         f"{len(ex)} of {len(INTER)} remedy×length×horizon cells have a 95% CI excluding 0")
else:
    gate("D2 remedy × backbone interaction", "NOT EVALUABLE", "no interaction table")
gate("D7 crossover length", "REPORTED" if len(CROSS) else "NOT EVALUABLE", f"{len(CROSS)} remedy×backbone×horizon rows")
if REPRO:
    _rr = pd.DataFrame(REPRO_ROWS); _cmp = _rr[_rr.status == "compared"] if len(_rr) and "status" in _rr else pd.DataFrame()
    gate("REPRODUCTION gate (clean-session rerun)", ("PASS" if len(_cmp) and _cmp.within_tol.all() else "FAIL") if len(_cmp) else "NOT EVALUABLE",
         f"{int(_cmp.within_tol.sum()) if len(_cmp) else 0} of {len(_cmp)} task×horizon RMSEs within ±{REPRO_TOL:.0%} of the reference ZIP")
    C("reproduction gate", len(_cmp) > 0 and bool(_cmp.within_tol.all()), f"{len(_cmp)} comparisons")
    SOFTWARE = "PASS" if all(c["result"] == "PASS" for c in chk) else "FAIL"; SW = pd.DataFrame(chk); save_csv(SW, DIRS["tables"] / "software_check.csv")
else:
    gate("REPRODUCTION gate (clean-session rerun)", "NOT RUN", "run after FULL with REPRO_REF_ZIP set to the FULL results ZIP")
if "SENS" in globals() and len(SENS):
    _sm = SENS[SENS.horizon_min == 15 * HORIZONS[0]]
    gate("Sensitivity: augmentation amount (0.25 / 1.00 vs 0.50)", "REPORTED",
         f"median RMSE change vs main {_sm.change_vs_main.median():+.1%} (range {_sm.change_vs_main.min():+.1%} to {_sm.change_vs_main.max():+.1%}); exploratory")
if REPLICATION_ONLY:                    # only H4 is tested in a replication-only run; H1-H3, D2, D7 come from the main FULL run
    G = [g for g in G if g["hypothesis"].startswith("H4")]
GATES = pd.DataFrame(G); save_csv(GATES, DIRS["tables"] / "research_gates.csv")
ELIGIBLE = MODE == "FULL_PAPER_RUN" and SOFTWARE == "PASS" and not REPRO

# ---- FEASIBILITY.md (Phase 0) ----
if len(FEAS):
    f = FEAS[FEAS.horizon_min == 15 * HORIZONS[0]]
    agg = f.headroom_aggregate.max(); tail = f.headroom_tail_top10pct_days.max()
    fv = "🟢 FEASIBLE-Q1" if (agg >= 0.20 or tail >= 0.15) else ("🟡 FEASIBLE-Q2" if (agg > 0 or tail > 0) else "🔴 REDESIGN")
    atomic_write_text(OUT / "FEASIBILITY.md", f"# Phase 0 feasibility ({MODE}, {label})\n\nVerdict: **{fv}** (aggregate headroom ≥ 0.20, i.e. 2× the 10% target, or tail headroom ≥ 0.15 → Q1)\n\n```\n{FEAS.to_string(index=False)}\n```\n")

# ---- manifest, hyperparameter ledger, hardware profile ----
hp = {}
for name, r in LEDGER.rows.items():
    if r["status"] == "COMPLETED" and r["kind"] == "model":
        z = load_task(name)
        if z is not None:
            hp[name] = z["meta"].get("hp", {})
save_json(dict(seeds=CFG["seeds"], lstm=dict(units=32, dense=8, dropout=0.2, batch=BATCH, lr_scratch=1e-3, lr_finetune=5e-4, updates_rule=f"max({CFG['e_base']} × ceil(n_real/{BATCH}), {CFG['u_min']})"),
               ddpm=dict(T=DDPM_T, beta=[1e-4, 0.02], schedule="linear", mlp=[128, 128], time_emb=64, iters=CFG["ddpm_iters"], batch=64, lr=1e-3),
               lgbm=dict(trials=CFG["lgbm_trials"], max_rounds=CFG["lgbm_rounds"], early_stopping=50), aug_frac=AUG_FRAC, jitter_sigma=JITTER_SIGMA,
               per_task=hp), DIRS["logs"] / "hyperparameter_ledger.json")
_net = LSTMNet(len(NN_STAT_IDX)).to(DEVICE); _x = _t(np.zeros((1000, LAGS))); _s = _t(np.zeros((1000, len(STAT_COLS))))
t0 = time.time()
with torch.no_grad():
    for _ in range(10):
        _net(_x, _s)
save_json(dict(lstm_params=n_params(_net), ddpm_params=n_params(EpsNet(LAGS + len(HORIZONS), len(NN_STAT_IDX))),
               lstm_inference_ms_per_1000=round((time.time() - t0) / 10 * 1000, 3), device=str(DEVICE),
               peak_gpu_mem_mb=round(torch.cuda.max_memory_allocated() / 1e6, 1) if DEVICE.type == "cuda" else None), DIRS["logs"] / "hardware_profiling.json")
save_json(dict(run_id=RUN_ID, mode=MODE, version=NOTEBOOK_VERSION, cfg_hash=CFG_HASH, data_hash=DATA_HASH, started=datetime.fromtimestamp(T0, timezone.utc).isoformat(),
               finished=datetime.now(timezone.utc).isoformat(), minutes=round((time.time() - T0) / 60, 2), preflight=PREFLIGHT, versions=VERSIONS,
               run_account=RUN_ACCOUNT, software=SOFTWARE, manuscript_eligible=ELIGIBLE), DIRS["logs"] / "run_manifest.json")
log_runtime("total", time.time() - T0)

# ---- RESULTS_AUDIT.md: claim → evidence ----
CLAIMS = [("R1 Phase 0 headroom", "tables/feasibility_headroom.csv"), ("R2 base-paper replication (H4)", "tables/replication_base_paper.csv"),
          ("R3 skill by arm × length × backbone", "tables/main_results.csv"), ("R4 DM tests, Holm, equivalence (H1, H2)", "tables/statistical_tests.csv"),
          ("R5 rank by length (H3)", "tables/rank_by_length.csv"), ("R6 subgroups", "tables/subgroup_results.csv"), ("R7 compute cost", "tables/compute_cost.csv"),
          ("R8 synthetic-data checks", "tables/synthetic_validity.csv"), ("R9 backbone interaction (D2)", "tables/interaction_backbone.csv"),
          ("R10 crossover length (D7)", "tables/crossover_length.csv"),
          ("R11 sensitivity: augmentation amount", "tables/sensitivity_aug_frac.csv"), ("R12 forecasts for recomputation", f"predictions/forecasts_{MODE}.npz"),
          ("R13 metric self-test", "tables/metric_self_test.csv")]
lines = [f"# RESULTS_AUDIT — {MODE} ({label})", "", f"SOFTWARE: **{SOFTWARE}** · MANUSCRIPT ELIGIBLE: **{ELIGIBLE}** · run ID `{RUN_ID}`", "",
         "| Result slot | Evidence file | Present |", "|---|---|---|"]
for c, f in CLAIMS:
    lines.append(f"| {c} | `{f}` | {'yes' if (OUT / f).exists() else 'NO — claim blocked'} |")
lines += ["", "## Research gates", GATES.to_string(index=False), "", "## Software check failures",
          SW[SW.result == "FAIL"].to_string(index=False) if (SW.result == "FAIL").any() else "none"]
if MODE == "SMOKE_TEST":
    lines.insert(1, "\n**NOT FOR PUBLICATION — smoke test on a small real subset; numbers check plumbing only.**\n")
atomic_write_text(OUT / "RESULTS_AUDIT.md", "\n".join(lines))

# ---- final verdict banner ----
light = {"PASS": "🟢", "FAIL": "🔴"}[SOFTWARE]
print("\n" + "=" * 78)
print(f"RUN ID: {RUN_ID}")
print(f"{light} SOFTWARE: {SOFTWARE}")
print(f"RESEARCH GATES ({label}):")
for r in GATES.itertuples():
    print(f"   {r.hypothesis}: {r.verdict}  —  {r.evidence}")
print(f"MANUSCRIPT ELIGIBLE: {ELIGIBLE}")
nxt = {"SMOKE_TEST": "STANDARD_RUN", "STANDARD_RUN": "FULL_PAPER_RUN", "FULL_PAPER_RUN": "manuscript (Stage 2)"}[MODE]
mins = (time.time() - T0) / 60
print(f"Runtime {mins:.1f} min on {PREFLIGHT['device']}. Next mode: {nxt}.")
print("=" * 78)

# ---- results ZIP (tables, figures, forecasts, logs, audits; no task files) ----
zp = export_results_zip()


## Limitations (stated before any result is read)
- One climate (arid Alice Springs) for the main benchmark; the Kaggle plants (India) serve replication only, because their location is not published (no clear-sky model or smart persistence for them).
- Point forecasts only; no probabilistic or coverage claims.
- Array rating, tilt and azimuth come from `array_metadata.csv` when supplied; otherwise rating is estimated from pre-origin data and tilt/azimuth default to 20°/north, recorded in `data_audit/array_selection.csv`.
- Physics pre-training uses a simple PVWatts-type chain (POA from measured GHI/DHI, NOCT-style cell temperature, −0.4%/°C, 4% losses); a better-calibrated chain could change A4.
- The LSTM keeps the base paper's size (32 units) on purpose; larger models may respond differently to augmentation.
- SMOKE_TEST and STANDARD_RUN results are exploratory; only a FULL_PAPER_RUN with SOFTWARE = PASS is manuscript-eligible.
